# Historical satellite extraction and field-data experiments

This notebook records the original Earth Engine experiments for the Vietnam crop-health project.
Its branches cover rice-grid covariates, Indonesia fire summaries, wheat/maize tests, provincial summaries, and field-level studies.
The cleanup preserves their calculations and code order as research history.

Use [replication/README.md](replication/README.md) for the maintained paper workflow.
Its Python modules include the recovered nine-band covariate branch, corrected bicubic upsampling, ESI/Aqua-only channels, crop-mask preparation, nightlights preparation, and detection-level fire reconstruction.
This notebook retains the historical calculations rather than adopting those later corrections silently.

## Reading and execution scope

Start with setup and shared helpers, then read the branch you need.
Annotations identify original one-based cell numbers at revision `19430e7`.
Later branches redefine `getNDVI`, `calculate_rvi`, `mask_crop_map`, dates, and geometries.
Their band names and masks depend on the selected branch's setup.
**Run All is not a single analysis recipe:** it would trigger authentication, uploads, exports, and repeated local writes across unrelated experiments.

## Requirements and files

The Python notebook kernel needs `earthengine-api`, `geemap`, `numpy`, `pandas`, `geopandas`, `shapely`, and `tqdm`.
SAR branches import the original `gee_s1_ard` Python files and `src.dhelper`.
The original Windows, mounted Drive, and Colab paths are preserved and need review before intentional execution.
Assigning `dir` does not change the working directory; relative paths resolve from the kernel's working directory.
The code assumes output subdirectories already exist.
Earth Engine assets require access through the user's account.
Upload cells reference files already stored in Cloud Storage, rather than creating those files.

| Branch | Required inputs | Historical outputs |
| --- | --- | --- |
| Rice covariates and NDVI | JAXA subtile JSON, `LULC_VN`, monthly `VNM_bm_YYYY_MM` assets | Drive folders `final_north`, `rice_leaf_south` |
| Indonesia fire summaries | GAUL Indonesia boundary and FIRMS T21 raster collection | Drive folder `FIRMS_IND` |
| Wheat/maize experiments | Full subtile JSON and WorldCereal assets | Local `wheat_south/`, `wheat_sar/` CSVs |
| Provincial study | GADM admin-1 GeoJSON and season windows | `datasets/seasonal_no_mask/` JSONs |
| EY field study | `datasets/Crop_Yield_Data_challenge_2.csv` | `datasets/EY/square_4w/` JSONs |
| ADB Thai Binh study | `datasets/thai_binh_insitu.csv` and GADM admin-2 boundaries | GeoJSONs, `datasets/thai_binh/` JSONs, Drive raster folders |

## Historical conventions and validation status

Monthly `STOP_DATE` is the last calendar day, although Earth Engine treats it as exclusive.
The monthly branches therefore omit that day's observations.
Rice extraction selects JAXA class `b1 == 3`; WorldCereal branches use `b1 == 100`.
The original rice code uses nearest-neighbor reprojection and no Sentinel-2 cloud mask.
Later field-level branches use separate cloud and range filters.
Tile slices and inconsistent branch labels are retained and explained beside the code.

No notebook code was executed during cleanup.
Format validation and Python/IPython syntax checks establish neither runtime success nor numerical reproduction.
Execution counts, progress displays, warnings, and stale HTML widgets were removed.
Two small saved input tables remain as labelled historical previews.
All nonempty code cells retain identical syntax trees; four Windows-path prefixes were clarified without changing their values.


## Setup

#### Import dependencies

Original cell 2.
Imports support tables, geometry, Earth Engine, and geemap.
Install them in the selected kernel environment before intentionally running setup.

In [ ]:
import os
import glob
import json
import sys
import datetime
from tqdm import tqdm_notebook
import math

import pandas as pd
import numpy as np
import geopandas as gpd
import geemap
from shapely import wkt

from shapely import Polygon

#### Initialize Earth Engine

Original cell 3.
This cell initializes project `ee-sonle96` and may open interactive authentication on failure.
The maintained command-line programs instead require authentication separately.

In [ ]:
import ee
# Trigger the Earth Engine authentication flow.
try:
    ee.Initialize(project='ee-sonle96')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='ee-sonle96')

#### Locate the original SAR modules

Original cell 4.
The module path points to a mounted Google Drive checkout.
Review it and the local `src.dhelper` import for your environment.
The `dir` assignment does not call `os.chdir`.

In [ ]:
# Adjust depending on your local setup
dir = 'G:/My Drive/Work/ADB/SAR/'
sys.path.append('G:/My Drive/Work/ADB/SAR/src/gee_s1_ard/python-api')
import border_noise_correction as bnc
import speckle_filter as sf
import terrain_flattening as trf
from src.dhelper import get_month_range, get_last_day_of_month_from_Ym, get_prev_ndays

## Historical Earth Engine asset uploads

#### Upload monthly nightlights

Original cell 7.
This loop matches local Colab filenames to existing Cloud Storage objects and submits them to `BM_VN_processed`.
Current preprocessing is in [prepare_nightlights.py](replication/prepare_nightlights.py).

In [ ]:
!earthengine authenticate

# Local paths for VNM Luminosity data 
paths = glob.glob('/content/gdrive/MyDrive/Work/WB/GPBP/data/VNM/luminosity/*.tif')

# Upload all valid assets to GEE
for path in paths[:]:
  ee_id = path[path.rindex('/')+1:path.rindex('.')]
  gcb_id = path[path.rindex('/')+1:]
  
  print(ee_id, gcb_id)
  !earthengine upload image --asset_id=projects/ee-sonle96/assets/BM_VN_processed/{ee_id} gs://adb-sar/black-marble-vn/{gcb_id}



#### Upload the JAXA mosaic

Original cell 8.
The cell uploads `LULC_VN_clipped.tif` rather than constructing it.
The author confirmed mosaicking Vietnam 2020 v23.09 tiles, and extraction below selects class 3.
See [prepare_crop_mask.py](replication/prepare_crop_mask.py).

In [ ]:
# Uploads JAXA LULC cropped to VNM's boundaries to EE 
!earthengine upload image --asset_id=projects/ee-sonle96/assets/LULC_VN gs://adb-sar/lulc/LULC_VN_clipped.tif

## Grid extraction branches

### Shared index definitions

#### Define shared indices

Original cell 11.
The initial RVI helper returns only `RVI_mean` from linear VV/VH power.
DPRVI uses a clipped VH/VV ratio, and EVI, LAI, and NDVI are separate optical calculations.
Later cells redefine some functions with different band names and masks.

In [ ]:
def calculate_rvi(image):
  """Calculates the Radar Vegetation Index (RVI) from a Sentinel-1 image using the formula in the paper below."""
  # https://www.sciencedirect.com/science/article/abs/pii/S0924271621001441?via%3Dihub

  n = image.select('VH').multiply(4)
  d = image.select('VV').add(image.select('VH'))
  rvi = n.divide(d).rename('RVI_mean')

  return rvi

def calculate_dprvi(image):
  # Calculates the RVI using the following formula:
  # https://github.com/Narayana-Rao/dual_pol_descriptors/blob/main/Jupyter%20Notebooks/1.Dual-pol%20descriptors.ipynb
  q = image.select('VH').divide(image.select('VV'))
  q = q.where(q.lt(0), 0)
  q = q.where(q.gt(1), 1)
  N = q.multiply(q.add(3))
  D = (q.add(1)).multiply(q.add(1))
  dprvi = N.divide(D).rename('DPRVI_mean')

  return dprvi

def calculate_vvvh(image):
  """Calculates the ratio of VV to VH bands."""
  vvvh = image.select('VV').divide(image.select('VH')).rename('VVVH')
  return vvvh

def calculate_vhvv(image):
  """Calculates the ratio of VH to VV bands."""
  vhvv = image.select('VH').divide(image.select('VV')).rename('VHVV')
  return vhvv


def getEVI(image):
    # Compute the EVI using an expression.
    EVI = image.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))', {
            'NIR': image.select('B8').divide(10000),
            'RED': image.select('B4').divide(10000),
            'BLUE': image.select('B2').divide(10000)
        }).rename("EVI")

    image = image.addBands(EVI)

    return(image)

def getLAI(image):
    LAI = image.expression(
        '(3.618*EVI - 0.118)', {
            'EVI': image.select('EVI')
        }).rename("LAI")
    image = image.addBands(LAI)

    return(image)

def getNDVI(image):
    
    # Normalized difference vegetation index (NDVI)
    ndvi = image.normalizedDifference(['B8','B4']).rename("NDVI")
    image = image.addBands(ndvi)

    return(image)



### First rice-grid settings

#### Set the first rice-grid period

Original cell 13.
This branch requests January-December 2017 at nominal 100 m in EPSG:4326.
The `MULTI` label does not select the filter: the next cell explicitly calls `MonoTemporal_Filter`.
JSON key order controls the later positional tile slice.

In [ ]:
# Loading JAXA subtiles for VNM
with open('shapefiles/JAXA_subtiles.json', 'r') as json_file:
    jaxa_subtiles = json.load(json_file)
tiles = list(jaxa_subtiles.keys())

# Paraneters for all images to be processed
crop_type = 'Rice'
country = 'VN'
crs = 'EPSG:4326'
scale=100
START_YM = '2017-01'
END_YM = '2017-12'
months = get_month_range(START_YM, END_YM, input_format='%Y-%m', output_format='%Y-%m')

# Parameters for SAR Processing
APPLY_BORDER_NOISE_CORRECTION = False
APPLY_TERRAIN_FLATTENING = True
APPLY_SPECKLE_FILTERING = True
POLARIZATION = 'VVVH'
PLATFORM_NUMBER = 'A'
ORBIT = 'DESCENDING'
ORBIT_NUM = None
SPECKLE_FILTER_FRAMEWORK = 'MULTI'
SPECKLE_FILTER = 'LEE SIGMA'
SPECKLE_FILTER_KERNEL_SIZE = 3 # 7
SPECKLE_FILTER_NR_OF_IMAGES = 3 # 10
TERRAIN_FLATTENING_MODEL = 'VOLUME'
DEM = ee.Image('USGS/SRTMGL1_003')
TERRAIN_FLATTENING_ADDITIONAL_LAYOVER_SHADOW_BUFFER = 0
CLIP_TO_ROI = True


### Legacy combined rice covariates

#### Legacy combined rice-covariate export

Original cell 15.
This annotated version includes NO2, O3, and wind in the sampled image.
Sentinel-5P coverage begins in 2018, so this is not the recovered 2017 production branch.
[extract_covariates.py](replication/extract_covariates.py) follows nine-band cell 120 at revision `4b8944d8f86c1b917bdf0f52397540ed9b10e0d8`.
The border-noise result assigned to `s1_1` is overwritten by filtering the original `s1`.
The `[33:]` tile slice and `final_north` folder are historical choices, not verified region labels.
PM2.5 is multiplied by `1e9`, temperature becomes Celsius, humidity remains a fraction, and precipitation remains in metres.
`Wind_direction_mean` has no computed band and is absent from the export selectors.

In [ ]:
print(f"CRS: {crs}")
print(f"scale: {scale}")
print(f"Start: {START_YM}. End: {END_YM}")

# Looping over all tiles
for tile in list(jaxa_subtiles.keys())[33:]: # [1:16] for NORTH

  print(f"  {tile}")
  bbox = jaxa_subtiles[tile]
  # Get EE Bounding Box object for the tile
  ROI = ee.Geometry.BBox(bbox[0],
                         bbox[1],
                         bbox[2],
                         bbox[3])\
                   .bounds(proj=crs)
  
  # JAXA LULC VN dataset
  # https://www.eorc.jaxa.jp/ALOS/en/dataset/lulc/lulc_vnm_v2309_e.htm
  jaxa = ee.Image('projects/ee-sonle96/assets/LULC_VN').clip(ROI).reproject('EPSG:4326', None, 10)
  mask_jaxa = jaxa.select('b1').eq(3) # b1 == 3 is the "rice" class

  def mask_crop_map(image):
    return image.updateMask(mask_jaxa)
                   
  # Looping over all months
  for month in months[:]:
    print(f"    {month}")

    # First and last days of the month
    START_DATE = month + '-01'
    STOP_DATE = get_last_day_of_month_from_Ym(month,
                                            input_format='%Y-%m',
                                            output_format='%Y-%m-%d')



    # Sentinel-1 GRD
    s1 = ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT')\
            .filter(ee.Filter.eq('instrumentMode', 'IW'))\
            .filter(ee.Filter.eq('resolution_meters', 10)) \
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))\
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))\
            .filter(ee.Filter.eq('platform_number', PLATFORM_NUMBER))\
            .filter(ee.Filter.eq('orbitProperties_pass', ORBIT))\
            .filterDate(START_DATE, STOP_DATE)\
            .select(['VV', 'VH', 'angle'])

      # Border noise correction
    s1_1 = s1.map(bnc.f_mask_edges)

      # Speckle filter
    s1_1 = ee.ImageCollection(sf.MonoTemporal_Filter(s1,
                                                    SPECKLE_FILTER_KERNEL_SIZE,
                                                    SPECKLE_FILTER))

      # Terrain correction
    s1_1 = (trf.slope_correction(s1_1,
                                TERRAIN_FLATTENING_MODEL,
                                DEM,
                                TERRAIN_FLATTENING_ADDITIONAL_LAYOVER_SHADOW_BUFFER))

      # RVI
    s1_1 = s1_1.map(calculate_rvi)
    rvi = s1_1.mean()\
              .updateMask(mask_jaxa)\
              .reproject(crs=crs, scale=scale)\
              .clip(ROI)\



    # CAMS PM2.5
    pm25 = ee.ImageCollection("ECMWF/CAMS/NRT")\
              .filter(ee.Filter.date(START_DATE, STOP_DATE))\
              .map(mask_crop_map)

    pm25_mean = pm25.select('particulate_matter_d_less_than_25_um_surface')\
                    .mean()\
                    .multiply(1E9)\
                    .rename('PM25_mean')\
                    .clip(ROI)\
                    .reproject(crs=crs, scale=scale)
    
    pm25_max = pm25.select('particulate_matter_d_less_than_25_um_surface')\
                    .max()\
                    .multiply(1E9)\
                    .rename('PM25_max')\
                    .clip(ROI)\
                    .reproject(crs=crs, scale=scale)  
    
    # Sentinel-5P
    no2 = ee.ImageCollection("COPERNICUS/S5P/OFFL/L3_NO2")\
            .filter(ee.Filter.date(START_DATE, STOP_DATE))\
            .map(mask_crop_map)\
            .filterBounds(ROI)
    no2_mean = no2.select('NO2_column_number_density')\
                  .mean()\
                  .rename('NO2_mean')\
                  .clip(ROI)\
                  .reproject(crs=crs, scale=scale)
    no2_max = no2.select('NO2_column_number_density')\
                  .max()\
                  .rename('NO2_max')\
                  .clip(ROI)\
                  .reproject(crs=crs, scale=scale)
    
    o3 = ee.ImageCollection("COPERNICUS/S5P/OFFL/L3_O3")\
            .filter(ee.Filter.date(START_DATE, STOP_DATE))\
            .map(mask_crop_map)\
            .filterBounds(ROI)
    o3_mean = o3.select('O3_column_number_density')\
                  .mean()\
                  .rename('O3_mean')\
                  .clip(ROI)\
                  .reproject(crs=crs, scale=scale)
    o3_max = o3.select('O3_column_number_density')\
                  .max()\
                  .rename('O3_max')\
                  .clip(ROI)\
                  .reproject(crs=crs, scale=scale)
    
    

    # ERA5
    era5 = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")\
              .filter(ee.Filter.date(START_DATE, STOP_DATE))\
              .map(mask_crop_map)

      # Average temperature
    mean_temp = era5.select('temperature_2m').mean()\
                  .subtract(273.15)\
                  .rename('Temperature_mean')\
                  .clip(ROI)\
                  .reproject(crs=crs, scale=scale)

      # Max temperature
    max_temp = era5.select('temperature_2m').max()\
                  .subtract(273.15)\
                  .rename('Temperature_max')\
                  .clip(ROI)\
                  .reproject(crs=crs, scale=scale)

      # wind speed (m/s) and direction as angle phi (radians)
      # https://confluence.ecmwf.int/pages/viewpage.action?pageId=133262398
    mean_wind_u = era5.select('u_component_of_wind_10m').mean()
    mean_wind_v = era5.select('v_component_of_wind_10m').mean()
    mean_wind_speed = ((mean_wind_u.multiply(mean_wind_u)).add(mean_wind_v.multiply(mean_wind_v))).sqrt()\
                                  .rename('Wind_speed_mean')\
                                  .clip(ROI)\
                                  .reproject(crs=crs, scale=scale)
    
    max_wind_u = era5.select('u_component_of_wind_10m').max()
    max_wind_v = era5.select('v_component_of_wind_10m').max()
    max_wind_speed = ((max_wind_u.multiply(max_wind_u)).add(max_wind_v.multiply(max_wind_v))).sqrt()\
                                  .rename('Wind_speed_max')\
                                  .clip(ROI)\
                                  .reproject(crs=crs, scale=scale)


    #   precipitation
    prep = era5.select('total_precipitation_sum').sum()\
              .rename('Precipitation_sum')\
              .clip(ROI)\
              .reproject(crs=crs, scale=scale)

      # Relative Humidity
      # https://earthscience.stackexchange.com/questions/16570/how-to-calculate-relative-humidity-from-temperature-dew-point-and-pressure?newreg=86cf2a3bb5bb47c5afd5fcaccfea796a
    b = 17.625
    c = 243.04
    mean_dew_temp = era5.select('dewpoint_temperature_2m').mean()\
                        .subtract(273.15)\
                        .rename('avg_dew_temperature')\
                        .clip(ROI)\
                        .reproject(crs=crs, scale=scale)

    num = mean_dew_temp.subtract(mean_temp).multiply(c).multiply(b)
    deno = (mean_temp.add(c)).multiply(mean_dew_temp.add(c))
    mean_rh = num.divide(deno).exp()\
                .rename('Rel_humidity_mean')

    
    # Black Marble
    bm = ee.Image(f"projects/ee-sonle96/assets/BM_VN_processed/VNM_bm_{month.replace('-', '_')}")\
          .select('b1')\
          .rename('Luminosity')\
          .clip(ROI)\
          .updateMask(mask_jaxa)\
          .reproject(crs=crs, scale=scale)
    
    # Elevation
    # negative values indicate elevation is below mean sea level  
    dem = ee.Image('USGS/SRTMGL1_003')\
            .select('elevation')\
            .updateMask(mask_jaxa)\
            .clip(ROI)\
            .reproject(crs=crs, scale=scale)
    

    # Combined
    combined = rvi.addBands(pm25_mean)\
                  .addBands(pm25_max)\
                  .addBands(no2_mean)\
                  .addBands(no2_max)\
                  .addBands(o3_mean)\
                  .addBands(o3_max)\
                  .addBands(mean_temp)\
                  .addBands(max_temp)\
                  .addBands(mean_wind_speed)\
                  .addBands(max_wind_speed)\
                  .addBands(prep)\
                  .addBands(mean_rh)\
                  .addBands(bm)\
                  .addBands(dem)

    # Transform image to feature collection
    feat_coll = combined.sample(factor=1,
                                region=ROI,
                                geometries=True,
                                scale=scale,
                                projection=crs)


    # Extract properties for each row
    def extract_properties(feature):
      properties = {
          'Timestamp': START_DATE,
          'lon': feature.geometry().coordinates().get(0),
          'lat': feature.geometry().coordinates().get(1),
          'CropType': crop_type,
          'Country': country,
          'RVI_mean': feature.get('RVI_mean'),
          'PM25_mean': feature.get('PM25_mean'),
          'PM25_max': feature.get('PM25_max'),
          'NO2_mean': feature.get('NO2_mean'),
          'NO2_max': feature.get('NO2_max'),
          'O3_mean': feature.get('O3_mean'),
          'O3_max': feature.get('O3_max'),
          'Temperature_mean': feature.get('Temperature_mean'),
          'Temperature_max': feature.get('Temperature_max'),
          'Wind_speed_mean': feature.get('Wind_speed_mean'),
          'Wind_direction_mean': feature.get('Wind_direction_mean'),
          'Wind_speed_max': feature.get('Wind_speed_max'),
          'Rel_humidity_mean': feature.get('Rel_humidity_mean'),
          'Precipitation_sum': feature.get('Precipitation_sum'),
          'Luminosity': feature.get('Luminosity'),
          'elevation': feature.get('elevation')
      }
      return ee.Feature(None, properties)

    processed_fc = feat_coll.map(extract_properties)

    # bands = ['Timestamp', 'lon', 'lat', 'CropType', 'Country',
    #       'RVI_mean', 'PM25_mean', 'PM25_max', 'NO2_mean', 'NO2_max',
    #       'O3_mean', 'O3_max', 'SO2_mean', 'SO2_max', 'Temperature_mean',
    #       'Temperature_max', 'Wind_speed_mean', 'Wind_direction_mean',
    #       'Wind_speed_max', 
    #       'Rel_humidity_mean', 'Precipitation_sum', 'Luminosity', 'elevation']
    
    # Bands to include in the export
    bands = ['Timestamp', 'lon', 'lat', 'CropType', 'Country',
             'RVI_mean', 'PM25_mean', 'PM25_max', 'Temperature_mean',
             'NO2_mean', 'NO2_max', 'O3_mean', 'O3_max',
             'Temperature_max', 'Rel_humidity_mean', 'Precipitation_sum', 
             'Wind_speed_mean', 'Wind_speed_max', 'Luminosity', 'elevation']
    
    # EE Batch export task 
    task = ee.batch.Export.table.toDrive(collection=processed_fc,
                                        description=f'{tile}_{scale}m_{START_DATE}',
                                        fileFormat='CSV',
                                        selectors=bands,
                                        folder='final_north')

    task.start()


## Legacy Indonesia fire summaries

#### Indonesia FIRMS T21 summaries

Original cell 20.
This branch exports country-level temporal/spatial brightness summaries for December 2016-December 2022 at nominal 1 km.
It does not produce detection-level FRP or T31 inputs.
Use [prepare_fire_detections.py](replication/prepare_fire_detections.py) for those inputs, or [extract_fire.py](replication/extract_fire.py) for the legacy summary workflow.

In [ ]:
crs = 'EPSG:4326'
scale=1000

START_YM = '2016-12'
END_YM = '2022-12'
months = get_month_range(START_YM, END_YM, input_format='%Y-%m', output_format='%Y-%m')

region = ee.FeatureCollection("FAO/GAUL/2015/level2").filter(ee.Filter.inList('ADM0_NAME', ['Indonesia'])).geometry()

for month in months:
    print(f"    {month}")

    START_DATE = month + '-01'
    STOP_DATE = get_last_day_of_month_from_Ym(month,
                                            input_format='%Y-%m',
                                            output_format='%Y-%m-%d')
                                            
    # FIRMS
    firms = ee.ImageCollection('FIRMS')\
            .filter(ee.Filter.date(START_DATE, STOP_DATE))\
            .select('T21')

    firms_mean = firms.mean().rename('Mean_T21')
    firms_max = firms.max().rename('Max_T21')

    fire_mean = firms_mean.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=region,
        scale=scale,
        crs=crs
    )

    fire_max = firms_max.reduceRegion(
        reducer=ee.Reducer.max(),
        geometry=region,
        scale=scale,
        crs=crs
    )

    properties = {
            'Timestamp': START_DATE,
            'Mean_T21': fire_mean.get('Mean_T21'),
            'Max_T21': fire_max.get('Max_T21')
        }
    export = ee.FeatureCollection(ee.Feature(None, properties))

    task = ee.batch.Export.table.toDrive(collection=export,
                                            description=f'IND_FIRMS_{START_DATE}',
                                            fileFormat='CSV',
                                            selectors=['Timestamp', 'Mean_T21', 'Max_T21'],
                                            folder='FIRMS_IND')

    task.start()

## Optical indices: crop comparisons and rice NDVI

#### Redefine optical helpers

Original cell 24.
Here NDVI returns only `NDVI_mean`, and LAI adds `LAI_mean` to the input image.
Those names differ from the first helper block.
The active rice export below emits NDVI summaries only.

In [ ]:
def getEVI(image):
    # Compute the EVI using an expression
    EVI = image.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))', {
            'NIR': image.select('B8').divide(10000),
            'RED': image.select('B4').divide(10000),
            'BLUE': image.select('B2').divide(10000)
        }).rename("EVI")

    image = image.addBands(EVI)

    return(image)

def getLAI(image):
    # Computes the LAI using an expression. The formula is from the paper below.
    # https://www.sciencedirect.com/science/article/pii/S0034425718302103
    LAI = image.expression(
        '(3.618*EVI - 0.118)', {
            'EVI': image.select('EVI')
        }).rename("LAI_mean")
    image = image.addBands(LAI)

    return(image)

def getNDVI(image):
    
    # Normalized difference vegetation index (NDVI)
    ndvi = image.normalizedDifference(['B8','B4']).rename("NDVI_mean")

    return(ndvi)

# def mask_wheat_map(img):
    
#     wheat_class_mask = img.select('classification').gt(50)
#     wheat_confi_mask = img.select('confidence').gt(50)
#     return img.updateMask(wheat_class_mask).updateMask(wheat_confi_mask)




#### Configure the wheat/maize comparison

Original cell 25.
Load the full tile mapping from the original Windows path and request 2017-2022.
The `Maize` crop label does not determine the actual mask used in the next branch.

In [ ]:
# JAXA Subtiles for VNM
with open(r"D:\Work\ADB\SAR\shapefiles\JAXA_subtiles_full.json", 'r') as json_file:
    jaxa_subtiles = json.load(json_file)

# Extraction parameters similar to the ones used in the Main IV extraction 
crop_type = 'Maize'
country = 'VN'
crs = 'EPSG:4326'
scale=100
START_YM = '2017-01'
END_YM = '2022-12'
months = get_month_range(START_YM, END_YM, input_format='%Y-%m', output_format='%Y-%m')

#### Select comparison tiles

Original cell 26.
The explicit historical list repeats `N09E105_1_0`.
That duplicate and its possible repeated writes are preserved.

In [ ]:
# chosen_tiles = [k for k in jaxa_subtiles.keys() if 'N10E105' in k] + [k for k in jaxa_subtiles.keys() if 'N09E105' in k] + ['N10E106_0_0', 'N10E106_0_1']
chosen_tiles = ['N10E105_0_1', 'N10E105_1_1', 'N10E106_0_1', 'N10E105_1_0','N09E105_1_0', 'N10E104_1_1', 'N09E105_0_0', 'N09E105_1_0', 'N09E105_0_1', 'N09E105_1_1', 'N10E105_0_0', 'N10E106_0_0']
# chosen_tiles = ['N09E106_0_0', 'N09E105_0_0', 'N09E105_0_0', 'N10E105_0_1', 'N10E105_0_0', 'N10E105_1_0',
#                 'N10E106_0_0', 'N10E106_1_0', 'N10E106_1_1']
chosen_tiles


#### Legacy mixed-mask covariates

Original cell 27.
NDVI, CAMS, and ERA5 use the wheat mask, while Black Marble and elevation use the maize mask.
Sampling their combined image requires joint validity across these masks.
The crop label says Maize and the folder says `wheat_south`; neither resolves the inconsistency.
NDVI property names contain trailing spaces.
This experiment is separate from the maintained rice pipeline.

In [ ]:
print(f"CRS: {crs}")
print(f"scale: {scale}")
print(f"Start: {START_YM}. End: {END_YM}")

# Winter Cereals (Wheat) calendar from WorldCereals
wheat = ee.Image('projects/ee-sonle96/assets/worldcereal_wintercereals_vn')
wheat_mask = wheat.select('b1').eq(100)

# Maize calendar from WorldCereals
maize = ee.Image('projects/ee-sonle96/assets/worldcereal_maize_vn')
maize_mask = maize.select('b1').eq(100)

# Function to apply the wheat calendar as a mask
def mask_crop_map(image):
    return image.updateMask(wheat_mask)

# Looping over subtiles 
for tile in chosen_tiles[:]:
    print(f"  {tile}")
    bbox = jaxa_subtiles[tile]
    ROI = ee.Geometry.BBox(bbox[0],
                            bbox[1],
                            bbox[2],
                            bbox[3])

    # Looping over months
    for month in months[:]:
        print(f"    {month}")

        START_DATE = month + '-01'
        STOP_DATE = get_last_day_of_month_from_Ym(month,
                                                input_format='%Y-%m',
                                                output_format='%Y-%m-%d')
        
        # https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_HARMONIZED
        # Sentinel-2 1C -> radiometrically calibrated
        ndvi = ee.ImageCollection('COPERNICUS/S2_HARMONIZED')\
            .filterDate(START_DATE, STOP_DATE)\
            .map(mask_crop_map)\
            .map(getNDVI)
        
        # https://kaflekrishna.com.np/blog-detail/calculating-ndvi-sentinel-2-images/
        ndvi_mean = ndvi.mean()\
                .clip(ROI)\
                .reproject(crs='EPSG:4326', scale=100)\
                
        ndvi_min = ndvi.min()\
                .rename('NDVI_min')\
                .clip(ROI)\
                .reproject(crs='EPSG:4326', scale=100)
        
        ndvi_max = ndvi.max()\
                .rename('NDVI_max')\
                .clip(ROI)\
                .reproject(crs='EPSG:4326', scale=100)
        
        pm25 = ee.ImageCollection("ECMWF/CAMS/NRT")\
              .filter(ee.Filter.date(START_DATE, STOP_DATE))\
              .map(mask_crop_map)

        pm25_mean = pm25.select('particulate_matter_d_less_than_25_um_surface')\
                        .mean()\
                        .multiply(1E9)\
                        .rename('PM25_mean')\
                        .clip(ROI)\
                        .reproject(crs='EPSG:4326', scale=100)
        
        pm25_min = pm25.select('particulate_matter_d_less_than_25_um_surface')\
                    .min()\
                    .multiply(1E9)\
                    .rename('PM25_min')\
                    .clip(ROI)\
                    .reproject(crs=crs, scale=scale)
        
        pm25_max = pm25.select('particulate_matter_d_less_than_25_um_surface')\
                    .max()\
                    .multiply(1E9)\
                    .rename('PM25_max')\
                    .clip(ROI)\
                    .reproject(crs=crs, scale=scale)
        
        # ERA5
        era5 = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")\
                .filter(ee.Filter.date(START_DATE, STOP_DATE))\
                .map(mask_crop_map)

        # Average temperature
        mean_temp = era5.select('temperature_2m').mean()\
                    .subtract(273.15)\
                    .rename('Temperature_mean')\
                    .clip(ROI)\
                    .reproject(crs=crs, scale=scale)

        # Max temperature
        max_temp = era5.select('temperature_2m').max()\
                    .subtract(273.15)\
                    .rename('Temperature_max')\
                    .clip(ROI)\
                    .reproject(crs=crs, scale=scale)

        #   precipitation
        prep = era5.select('total_precipitation_sum').sum()\
                .rename('Precipitation_sum')\
                .clip(ROI)\
                .reproject(crs=crs, scale=scale)

        # Relative Humidity
        # https://earthscience.stackexchange.com/questions/16570/how-to-calculate-relative-humidity-from-temperature-dew-point-and-pressure?newreg=86cf2a3bb5bb47c5afd5fcaccfea796a
        b = 17.625
        c = 243.04
        mean_dew_temp = era5.select('dewpoint_temperature_2m').mean()\
                            .subtract(273.15)\
                            .rename('avg_dew_temperature')\
                            .clip(ROI)\
                            .reproject(crs=crs, scale=scale)

        num = mean_dew_temp.subtract(mean_temp).multiply(c).multiply(b)
        deno = (mean_temp.add(c)).multiply(mean_dew_temp.add(c))
        mean_rh = num.divide(deno).exp()\
                    .rename('Rel_humidity_mean')

        
        # Black Marble
        bm = ee.Image(f"projects/ee-sonle96/assets/BM_VN_processed/VNM_bm_{month.replace('-', '_')}")\
            .select('b1')\
            .rename('Luminosity')\
            .clip(ROI)\
            .reproject(crs=crs, scale=scale)\
            .updateMask(maize_mask)
            
        
        # Elevation
        # negative values indicate elevation is below mean sea level  
        dem = ee.Image('USGS/SRTMGL1_003')\
                .select('elevation')\
                .clip(ROI)\
                .reproject(crs=crs, scale=scale)\
                .updateMask(maize_mask)
        
        combined = ndvi_mean.addBands(ndvi_min)\
                            .addBands(ndvi_max)\
                            .addBands(pm25_mean)\
                            .addBands(pm25_min)\
                            .addBands(pm25_max)\
                            .addBands(mean_temp)\
                            .addBands(max_temp)\
                            .addBands(prep)\
                            .addBands(mean_rh)\
                            .addBands(bm)\
                            .addBands(dem)
        
        feat_coll = combined.sample(factor=1,
                                    region=ROI,
                                    geometries=True,
                                    scale=100,
                                    projection='EPSG:4326')


        def extract_properties(feature):
            properties = {
                'Timestamp': START_DATE,
                'lon': feature.geometry().coordinates().get(0),
                'lat': feature.geometry().coordinates().get(1),
                'CropType': crop_type,
                'NDVI_mean ': feature.get('NDVI_mean'),
                'NDVI_min ': feature.get('NDVI_min'),
                'NDVI_max ': feature.get('NDVI_max'),
                'PM25_mean': feature.get('PM25_mean'),
                'PM25_min': feature.get('PM25_min'),
                'PM25_max': feature.get('PM25_max'),
                'Temperature_mean': feature.get('Temperature_mean'),
                'Temperature_max': feature.get('Temperature_max'),
                'Rel_humidity_mean': feature.get('Rel_humidity_mean'),
                'Precipitation_sum': feature.get('Precipitation_sum'),
                'Luminosity': feature.get('Luminosity'),
                'elevation': feature.get('elevation')
            }
            return ee.Feature(None, properties)

        processed_fc = feat_coll.map(extract_properties)
        
        # bands = ['Timestamp', 'lon', 'lat', 'CropType', 'Country',
        #         'NDVI_mean', 'PM25_mean']
        
        # task = ee.batch.Export.table.toDrive(collection=processed_fc,
        #                                     description=f'{tile}_{scale}m_{START_DATE}',
        #                                     fileFormat='CSV',
        #                                     selectors=bands,
        #                                     folder='wheat_south')

        # task.start()
        
        # Alternative way to extract the data
        df = geemap.ee_to_df(processed_fc)
        
        # Saving as CSVs locally 
        df.to_csv(rf'D:\Work\ADB\SAR\wheat_south\{tile}_{scale}m_{START_DATE}.csv', index=False)



#### Reset rice NDVI settings

Original cell 29.
This cell requests 2017-2022 and reloads the selected rice tile mapping.
Its displayed slice is a preview; the next cell selects its own loop slice.

In [ ]:
START_YM = '2017-01'
END_YM = '2022-12'
months = get_month_range(START_YM, END_YM, input_format='%Y-%m', output_format='%Y-%m')

crs = 'EPSG:4326'
scale = 100
# Loading JAXA subtiles for VNM
with open('shapefiles/JAXA_subtiles.json', 'r') as json_file:
    jaxa_subtiles = json.load(json_file)
tiles = list(jaxa_subtiles.keys())
tiles[:16]

#### Historical rice NDVI exports

Original cell 30.
The active code uses Sentinel-2 Harmonized Level-1C B8/B4 NDVI, JAXA class 3, and monthly mean/minimum/maximum.
It applies no cloud mask.
The `[1:16]` slice and `rice_leaf_south` folder are preserved historical choices.
NDVI export property names include trailing spaces; [extract_ndvi.py](replication/extract_ndvi.py) uses explicit clean selectors.

In [ ]:
print(f"CRS: {crs}")
print(f"scale: {scale}")
print(f"Start: {START_YM}. End: {END_YM}")

# Loading JAXA subtiles for VNM
with open('shapefiles/JAXA_subtiles.json', 'r') as json_file:
    jaxa_subtiles = json.load(json_file)
tiles = list(jaxa_subtiles.keys())

# Looping over subtiles 
for tile in list(jaxa_subtiles.keys())[1:16]: # [1:16] for SOUTH
    print(f"  {tile}")
    bbox = jaxa_subtiles[tile]
    ROI = ee.Geometry.BBox(bbox[0],
                            bbox[1],
                            bbox[2],
                            bbox[3])
    
      # JAXA LULC VN dataset
    # https://www.eorc.jaxa.jp/ALOS/en/dataset/lulc/lulc_vnm_v2309_e.htm
    jaxa = ee.Image('projects/ee-sonle96/assets/LULC_VN').clip(ROI).reproject('EPSG:4326', None, 10)
    mask_jaxa = jaxa.select('b1').eq(3) # b1 == 3 is the "rice" class

    def mask_crop_map(image):
        return image.updateMask(mask_jaxa)

    # Looping over months
    for month in months[:]:
        print(f"    {month}")

        START_DATE = month + '-01'
        STOP_DATE = get_last_day_of_month_from_Ym(month,
                                                input_format='%Y-%m',
                                                output_format='%Y-%m-%d')
        
        # # Sentinel-1 GRD
        # s1 = ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT')\
        #         .filter(ee.Filter.eq('instrumentMode', 'IW'))\
        #         .filter(ee.Filter.eq('resolution_meters', 10)) \
        #         .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))\
        #         .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))\
        #         .filter(ee.Filter.eq('platform_number', PLATFORM_NUMBER))\
        #         .filter(ee.Filter.eq('orbitProperties_pass', ORBIT))\
        #         .filterDate(START_DATE, STOP_DATE)\
        #         .select(['VV', 'VH', 'angle'])

        # # Border noise correction
        # s1_1 = s1.map(bnc.f_mask_edges)

        # # Speckle filter
        # s1_1 = ee.ImageCollection(sf.MonoTemporal_Filter(s1,
        #                                                 SPECKLE_FILTER_KERNEL_SIZE,
        #                                                 SPECKLE_FILTER))

        # # Terrain correction
        # s1_1 = (trf.slope_correction(s1_1,
        #                             TERRAIN_FLATTENING_MODEL,
        #                             DEM,
        #                             TERRAIN_FLATTENING_ADDITIONAL_LAYOVER_SHADOW_BUFFER))

        # # RVI
        # s1_1 = s1_1.map(calculate_rvi)
        # rvi = s1_1.mean()\
        #         .updateMask(mask_jaxa)\
        #         .reproject(crs=crs, scale=scale)\
        #         .clip(ROI)\

        # # ERA5
        # era5 = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")\
        #         .filter(ee.Filter.date(START_DATE, STOP_DATE))\
        #         .map(mask_crop_map)

        # # Average temperature
        # mean_temp = era5.select('temperature_2m').mean()\
        #             .subtract(273.15)\
        #             .rename('Temperature_mean')\
        #             .clip(ROI)\
        #             .reproject(crs=crs, scale=scale)
        
        # https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_HARMONIZED
        # Sentinel-2 1C -> radiometrically calibrated
        ndvi = ee.ImageCollection('COPERNICUS/S2_HARMONIZED')\
            .filter(ee.Filter.date(START_DATE, STOP_DATE))\
            .map(mask_crop_map)\
            .map(getNDVI)
        
        # https://kaflekrishna.com.np/blog-detail/calculating-ndvi-sentinel-2-images/
        ndvi_mean = ndvi.mean()\
                .reproject(crs=crs, scale=scale)\
                
        ndvi_min = ndvi.min()\
                .rename('NDVI_min')\
                .reproject(crs=crs, scale=scale)
        
        ndvi_max = ndvi.max()\
                .rename('NDVI_max')\
                .reproject(crs=crs, scale=scale)
        
        
        combined = ndvi_mean.addBands(ndvi_min)\
                            .addBands(ndvi_max)
        
        # combined = mean_temp.addBands(ndvi_mean)\
        #               .addBands(ndvi_min)\
        #               .addBands(ndvi_max)
        
        feat_coll = combined.sample(factor=1,
                                    region=ROI,
                                    geometries=True,
                                    scale=100,
                                    projection='EPSG:4326')


        def extract_properties(feature):
            properties = {
                'Timestamp': START_DATE,
                'lon': feature.geometry().coordinates().get(0),
                'lat': feature.geometry().coordinates().get(1),
                # 'Temperature_mean': feature.get('Temperature_mean'),
                'NDVI_mean ': feature.get('NDVI_mean'),
                'NDVI_min ': feature.get('NDVI_min'),
                'NDVI_max ': feature.get('NDVI_max')
            }
            return ee.Feature(None, properties)

        processed_fc = feat_coll.map(extract_properties)
        
        # bands = ['Timestamp', 'lon', 'lat', 'NDVI_mean', 'NDVI_min', 'NDVI_max']
        
        task = ee.batch.Export.table.toDrive(collection=processed_fc,
                                            description=f'{tile}_{scale}m_{START_DATE}',
                                            fileFormat='CSV',
                                            folder='rice_leaf_south')

        task.start()
        
        # Alternative way to extract the data
        # df = geemap.ee_to_df(processed_fc)
        
        # Saving as CSVs locally 
        # df.to_csv(f'D:\Work\ADB\SAR\leaf_south\{tile}_{scale}m_{START_DATE}.csv', index=False)





## Legacy wheat SAR summaries

#### Redefine SAR helper outputs

Original cell 34.
This `calculate_rvi` adds `RVI` to the input image instead of returning only `RVI_mean`.
It resets 2017-2022 parameters and the tile list for the following wheat branch.

In [ ]:
def calculate_rvi(image):

  n = image.select('VH').multiply(4)
  d = image.select('VV').add(image.select('VH'))
  rvi = n.divide(d).rename('RVI')

  return image.addBands(rvi, None, True)

def calculate_vvvh(image):
  """Calculates the ratio of VV to VH bands."""
  vvvh = image.select('VV').divide(image.select('VH')).rename('VVVH')
  return vvvh

def calculate_vhvv(image):
  """Calculates the ratio of VH to VV bands."""
  vhvv = image.select('VH').divide(image.select('VV')).rename('VHVV')
  return vhvv

# Parameters to filter images from the Sentinel-1 GRD collection
crs = 'EPSG:4326'
scale=100
START_YM = '2017-01'
END_YM = '2022-12'
months = get_month_range(START_YM, END_YM, input_format='%Y-%m', output_format='%Y-%m')
APPLY_BORDER_NOISE_CORRECTION = False
APPLY_TERRAIN_FLATTENING = True
APPLY_SPECKLE_FILTERING = True
POLARIZATION = 'VVVH'
PLATFORM_NUMBER = 'A'
ORBIT = 'DESCENDING'
ORBIT_NUM = None
SPECKLE_FILTER_FRAMEWORK = 'SINGLE'
SPECKLE_FILTER = 'LEE SIGMA'
SPECKLE_FILTER_KERNEL_SIZE = 3 # 7
SPECKLE_FILTER_NR_OF_IMAGES = 2 # 10
TERRAIN_FLATTENING_MODEL = 'VOLUME'
DEM = ee.Image('USGS/SRTMGL1_003')
TERRAIN_FLATTENING_ADDITIONAL_LAYOVER_SHADOW_BUFFER = 0
CLIP_TO_ROI = True

# JAXA Subtiles for VNM
with open(r"D:\Work\ADB\SAR\shapefiles\JAXA_subtiles_full.json", 'r') as json_file:
    jaxa_subtiles = json.load(json_file)

chosen_tiles = ['N10E105_0_1', 'N10E105_1_1', 'N10E106_0_1', 'N10E105_1_0','N09E105_1_0', 'N10E104_1_1', 'N09E105_0_0', 'N09E105_1_0', 'N09E105_0_1', 'N09E105_1_1', 'N10E105_0_0', 'N10E106_0_0']
# chosen_tiles = ['N09E106_0_0', 'N09E105_0_0', 'N09E105_0_0', 'N10E105_0_1', 'N10E105_0_0', 'N10E105_1_0',
#                 'N10E106_0_0', 'N10E106_1_0', 'N10E106_1_1']    


#### Wheat SAR summaries

Original cell 35.
The active mask selects WorldCereal wheat, and the combined image contains 15 monthly RVI/VV/VH/ratio summaries.
Each table is fetched with `geemap.ee_to_df` and saved under `wheat_sar`.
Exceptions print `failed` and continue, so loop completion does not prove complete output coverage.
The optional summaries in [extract_rvi.py](replication/extract_rvi.py) use these definitions with an explicit rice mask.

In [ ]:
print(f"CRS: {crs}")
print(f"scale: {scale}")
print(f"Start: {START_YM}. End: {END_YM}")

wheat = ee.Image('projects/ee-sonle96/assets/worldcereal_wintercereals_vn')
wheat_mask = wheat.select('b1').eq(100)

maize = ee.Image('projects/ee-sonle96/assets/worldcereal_maize_vn')
maize_mask = maize.select('b1').eq(100)

crop_type = 'Wheat'

def mask_crop_map(image):
    return image.updateMask(wheat_mask)

for tile in chosen_tiles[:]:
    print(f"  {tile}")
    bbox = jaxa_subtiles[tile]
    ROI = ee.Geometry.BBox(bbox[0],
                            bbox[1],
                            bbox[2],
                            bbox[3])

    for month in months[:]:
        print(f"    {month}")

        START_DATE = month + '-01'
        STOP_DATE = get_last_day_of_month_from_Ym(month,
                                                input_format='%Y-%m',
                                                output_format='%Y-%m-%d')
        
        # RVI
        s1 = ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT')\
                .filter(ee.Filter.eq('instrumentMode', 'IW'))\
                .filter(ee.Filter.eq('resolution_meters', 10)) \
                .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))\
                .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))\
                .filter(ee.Filter.eq('platform_number', PLATFORM_NUMBER))\
                .filter(ee.Filter.eq('orbitProperties_pass', ORBIT))\
                .filterDate(START_DATE, STOP_DATE)\
                .select(['VV', 'VH', 'angle'])

        # Speckle filter
        s1_1 = ee.ImageCollection(sf.MonoTemporal_Filter(s1,
                                                        SPECKLE_FILTER_KERNEL_SIZE,
                                                        SPECKLE_FILTER))

        # Terrain correction
        s1_1 = (trf.slope_correction(s1_1,
                                    TERRAIN_FLATTENING_MODEL,
                                    DEM,
                                    TERRAIN_FLATTENING_ADDITIONAL_LAYOVER_SHADOW_BUFFER))

        # RVI
        s1_1 = s1_1.map(calculate_rvi)\
                   .map(mask_crop_map)
        


        rvi_mean = s1_1.mean()\
                .select("RVI")\
                .rename("RVI_mean")\
                # .reproject(crs=crs, scale=scale)\
                # .clip(ROI)\
                
                
        rvi_min = s1_1.min()\
                .select("RVI")\
                .rename("RVI_min")\
                # .reproject(crs=crs, scale=scale)\
                # .clip(ROI)\
                
                
        rvi_max = s1_1.max()\
                .select("RVI")\
                .rename("RVI_max")\
                # .reproject(crs=crs, scale=scale)\
                # .clip(ROI)\
                
        
        vv_mean = s1_1.mean()\
                .select("VV")\
                .rename("VV_mean")\
                # .reproject(crs=crs, scale=scale)\
                # .clip(ROI)\
        
        vv_min = s1_1.min()\
                .select("VV")\
                .rename("VV_min")\
                # .reproject(crs=crs, scale=scale)\
                # .clip(ROI)\
                
        vv_max = s1_1.max()\
                .select("VV")\
                .rename("VV_max")\
                # .reproject(crs=crs, scale=scale)\
                # .clip(ROI)\
                
        vh_mean = s1_1.mean()\
                .select("VH")\
                .rename("VH_mean")\
                # .reproject(crs=crs, scale=scale)\
                # .clip(ROI)\
        
        vh_min = s1_1.min()\
                .select("VH")\
                .rename("VH_min")\
                # .reproject(crs=crs, scale=scale)\
                # .clip(ROI)\
                
        vh_max = s1_1.max()\
                .select("VH")\
                .rename("VH_max")\
                # .reproject(crs=crs, scale=scale)\
                # .clip(ROI)\
                
        vvvh_mean = s1_1.map(calculate_vvvh)\
                        .mean()\
                        .select("VVVH")\
                        .rename("VVVH_mean")\
                        # .reproject(crs=crs, scale=scale)\
                        # .clip(ROI)\
                        
        
        vvvh_min = s1_1.map(calculate_vvvh)\
                        .min()\
                        .select("VVVH")\
                        .rename("VVVH_min")\
                        # .reproject(crs=crs, scale=scale)\
                        # .clip(ROI)\
        
        vvvh_max = s1_1.map(calculate_vvvh)\
                        .max()\
                        .select("VVVH")\
                        .rename("VVVH_max")\
                        # .reproject(crs=crs, scale=scale)\
                        # .clip(ROI)\
                        
        vhvv_mean = s1_1.map(calculate_vhvv)\
                        .mean()\
                        .select("VHVV")\
                        .rename("VHVV_mean")\
                        # .reproject(crs=crs, scale=scale)\
                        # .clip(ROI)\
                        
        
        vhvv_min = s1_1.map(calculate_vhvv)\
                        .min()\
                        .select("VHVV")\
                        .rename("VHVV_min")\
                        # .reproject(crs=crs, scale=scale)\
                        # .clip(ROI)\
        
        vhvv_max = s1_1.map(calculate_vhvv)\
                        .max()\
                        .select("VHVV")\
                        .rename("VHVV_max")\
                        # .reproject(crs=crs, scale=scale)\
                        # .clip(ROI)\
                
        
        
        combined = rvi_mean.addBands(rvi_min)\
                            .addBands(rvi_max)\
                            .addBands(vv_mean)\
                            .addBands(vv_min)\
                            .addBands(vv_max)\
                            .addBands(vh_mean)\
                            .addBands(vh_min)\
                            .addBands(vh_max)\
                            .addBands(vvvh_mean)\
                            .addBands(vvvh_min)\
                            .addBands(vvvh_max)\
                            .addBands(vhvv_mean)\
                            .addBands(vhvv_min)\
                            .addBands(vhvv_max)\
                            
        
        feat_coll = combined.sample(factor=1,
                                    region=ROI,
                                    geometries=True,
                                    scale=scale,
                                    projection=crs)


        def add_properties(feature):
            
            return feature.set({
                'Timestamp': START_DATE,
                'lon': feature.geometry().coordinates().get(0),
                'lat': feature.geometry().coordinates().get(1)
            })
        
        processed_fc = feat_coll.map(add_properties)
        
        # bands = ['Timestamp', 'lon', 'lat', 'CropType', 'RVI_mean',
        #         'RVI_min', 'RVI_max']
        
        # task = ee.batch.Export.table.toDrive(collection=processed_fc,
        #                                     description=f'{tile}_{scale}m_{START_DATE}_wheat',
        #                                     fileFormat='CSV',
        #                                     selectors=['Timestamp', 'lon', 'lat', 'RVI_mean'],
        #                                     folder='wheat_south')

        # task.start()
        try:
            df = geemap.ee_to_df(processed_fc)

            df.to_csv(rf'D:\Work\ADB\SAR\wheat_sar\{tile}_{scale}m_{START_DATE}.csv', index=False)
        except Exception as e:
            print('     failed')
            continue

## Legacy provincial seasonal study

#### Inspect provincial boundaries

Original cell 42.
The saved five-row GADM table is a historical input preview, retained without rerunning.
`names` is sorted independently of `poly`; pairing sorted IDs with original row positions can mislabel geometries unless both orders coincide.

In [ ]:
# Loading VNM's Provincial shapefile from GADM
poly = gpd.read_file('shapefiles/gadm41_VNM_1.json')
names = poly['GID_1'].to_list()
names.sort()

poly.head()

,GID_1,GID_0,COUNTRY,NAME_1,VARNAME_1,NL_NAME_1,TYPE_1,ENGTYPE_1,CC_1,HASC_1,ISO_1,geometry
0,VNM.1_1,VNM,Vietnam,AnGiang,AnGiang,NA,Tỉnh,Province,NA,VN.AG,VN-44,"MULTIPOLYGON (((105.54860 10.42950, 105.54950 ..."
1,VNM.7_1,VNM,Vietnam,BàRịa-VũngTàu,BaRia-VungTau,NA,Tỉnh,Province,NA,VN.BV,NA,"MULTIPOLYGON (((107.09010 10.32400, 107.08890 ..."
2,VNM.3_1,VNM,Vietnam,BắcGiang,BacGiang,NA,Tỉnh,Province,NA,VN.BG,NA,"MULTIPOLYGON (((106.28380 21.13230, 106.27340 ..."
3,VNM.4_1,VNM,Vietnam,BắcKạn,BacKan,NA,Tỉnh,Province,NA,VN.BK,NA,"MULTIPOLYGON (((105.87240 21.85580, 105.86290 ..."
4,VNM.2_1,VNM,Vietnam,BạcLiêu,BacLieu,NA,Tỉnh,Province,NA,VN.BL,NA,"MULTIPOLYGON (((105.42440 9.02130, 105.41640 9..."


#### Define exploratory seasons

Original cell 43.
These windows are historical study settings, not the paper reproductive-stage assignment.
The preserved September entries end on invalid date `09-31`.
Resolve their intended end dates before selecting those entries.

In [ ]:
# Dictionary of rice planting seasons in the north and south of Vietnam
seasons = {'early_spring': ['{}-01-15', '{}-02-15'],
           'late_spring': ['{}-05-01', '{}-05-31'],
           'main_autumn': ['{}-06-01', '{}-07-31'],
           'late_autumn': ['{}-06-15', '{}-12-15'],
           'winter': ['{}-11-01', '{}-11-30'],
           'rice_1_s': ['{}-01-01', '{}-01-31'],
           'rice_2_s': ['{}-07-01', '{}-07-31'],
           'rice_3_s': ['{}-09-01', '{}-09-31'],
           'rice_1_n': ['{}-09-01', '{}-09-31'],
           'rice_2_n': ['{}-07-01', '{}-07-31'],
           'rice_3_n': ['{}-10-01', '{}-10-30']}

#### Provincial seasonal summaries

Original cell 44.
The active `early_spring` branch calculates province means for 2016-2022 without a crop mask and writes under `datasets/seasonal_no_mask`.
It depends on the current helper definitions, including the changed optical band names.
This is separate from plot-month paper extraction.

In [ ]:
season = 'early_spring'
print(season)

# Looping over years 
for year in list(range(2016, 2023, 1)):
    print(f'  {year}')
    START_DATE = seasons[season][0].format(str(year))
    STOP_DATE = seasons[season][1].format(str(year))

    # Looping over provinces
    for i in range(len(poly)):
      
      r = poly.iloc[i:i+1]
      name = names[i]
      print(f'    {name}')
      # Transforming GeoDataFrame elements to Earth Engine geometries 
      regions = geemap.gdf_to_ee(r).geometry()



      # RVI
      s1 = ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT')\
              .filter(ee.Filter.eq('instrumentMode', 'IW'))\
              .filter(ee.Filter.eq('resolution_meters', 10)) \
              .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))\
              .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))\
              .filter(ee.Filter.eq('platform_number', PLATFORM_NUMBER))\
              .filter(ee.Filter.eq('orbitProperties_pass', ORBIT))\
              .filterDate(START_DATE, STOP_DATE)\
              .select(['VV', 'VH', 'angle'])


        # Speckle filter
      s1_1 = ee.ImageCollection(sf.MonoTemporal_Filter(s1,
                                                      SPECKLE_FILTER_KERNEL_SIZE,
                                                      SPECKLE_FILTER))

      # Terrain correction
      s1_1 = (trf.slope_correction(s1_1,
                                  TERRAIN_FLATTENING_MODEL,
                                  DEM,
                                  TERRAIN_FLATTENING_ADDITIONAL_LAYOVER_SHADOW_BUFFER))

      # S2 
      # https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_HARMONIZED
      # Sentinel-2 1C -> radiometrically calibrated
      s2 = ee.ImageCollection('COPERNICUS/S2_HARMONIZED')\
            .filterDate(START_DATE, STOP_DATE)\
      # https://kaflekrishna.com.np/blog-detail/retrieving-leaf-area-index-lai-sentinel-2-image-google-earth-engine-gee/
      lai = s2.map(getEVI)\
              .map(getLAI)\
              .mean()\
      
      # https://kaflekrishna.com.np/blog-detail/calculating-ndvi-sentinel-2-images/
      nvdi = s2.map(getNDVI)\
              .mean()




      dprvi = s1_1.map(calculate_dprvi)\
                .mean()\
                # .updateMask(mask_jaxa)

      rvi = s1_1.map(calculate_rvi)\
                .mean()\
                # .updateMask(mask_jaxa)
      
      vvvh = s1_1.map(calculate_vvvh)\
                .mean()\
                # .updateMask(mask_jaxa)
      

      vhvv = s1_1.map(calculate_vhvv)\
                .mean()\
                # .updateMask(mask_jaxa)
      
      img = s1_1.mean()\
                # .updateMask(mask_jaxa)
      
      combined = img.addBands(dprvi)\
                    .addBands(rvi)\
                    .addBands(vvvh)\
                    .addBands(vhvv)\
                    .addBands(nvdi)\
                    .addBands(lai)


      stats = combined.reduceRegion(
                      geometry=regions,
                      reducer=ee.Reducer.mean(),
                      scale=100,  
                      crs='EPSG:4326', 
                      tileScale=14,
                      bestEffort=True
                )

      # Exporting the data locallyh as JSON files
      try:
        test_d = stats.getInfo()
        test_d['province_id'] = name
        test_d['date'] = START_DATE
        test_d['season'] = season
        with open(f'datasets/seasonal_no_mask/{season}/{name}_{year}.json', 'w') as out_file:
          json.dump(test_d, out_file)
      except Exception as e:
         print(e)
         continue


      

## Separate field-data experiments

### EY crop-yield study

#### Load EY field observations

Original cell 49.
The saved five-row table is a historical input preview, not a result of this cleanup.
The cell converts hectares to square metres and derives nominal radii and side lengths.

In [ ]:
# Loading and processing the EY dataset
dataset = pd.read_csv('datasets/Crop_Yield_Data_challenge_2.csv')
dataset['field_size_m2'] = dataset['Field size (ha)'] * 10000
dataset['radius'] = np.sqrt(dataset['field_size_m2'] / np.pi)
dataset['side'] = np.sqrt(dataset['field_size_m2'])
dataset['id'] = dataset.index

# Extracting rice fields data 
lats = dataset['Latitude'].to_list()
lons = dataset['Longitude'].to_list()
dates = dataset['Date of Harvest'].to_list()
radii = dataset['radius'].to_list()
sides = dataset['side'].to_list()

dataset.head()

,District,Latitude,Longitude,"Season(SA = Summer Autumn, WS = Winter Spring)","Rice Crop Intensity(D=Double, T=Triple)",Date of Harvest,Field size (ha),Rice Yield (kg/ha),field_size_m2,radius,side,id
0,Chau_Phu,10.510542,105.248554,SA,T,15-07-2022,3.40,5500,34000.0,104.031419,184.390889,0
1,Chau_Phu,10.509150,105.265098,SA,T,15-07-2022,2.43,6000,24300.0,87.948452,155.884573,1
2,Chau_Phu,10.467721,105.192464,SA,D,15-07-2022,1.95,6400,19500.0,78.784788,139.642400,2
3,Chau_Phu,10.494453,105.241281,SA,T,15-07-2022,4.30,6000,43000.0,116.992842,207.364414,3
4,Chau_Phu,10.535058,105.252744,SA,D,14-07-2022,3.30,6400,33000.0,102.490128,181.659021,4


#### Construct EY buffers

Original cell 50.
Buffers are constructed in EPSG:3857 and transformed back to EPSG:4326.
Both circle and square calls use `sides[i]` as buffer distance rather than the calculated radius.
Keep that distinction when interpreting polygon areas.

In [ ]:
def create_poly(lat, lon, distance, type='round'):
    """Creates buffer polygons around a point"""
    gs = gpd.GeoSeries(wkt.loads(f'POINT ({lon} {lat})'))
    gdf = gpd.GeoDataFrame(geometry=gs)
    gdf.crs='EPSG:4326'
    gdf = gdf.to_crs('EPSG:3857')
    res = gdf.buffer(
        distance=distance,
        cap_style=type,
    )

    return res.to_crs('EPSG:4326').iloc[0]

# Creating circular and square polygons around the rice fields
circles = []
squares = []

for i in tqdm_notebook(range(len(lats))):
    circles.append(create_poly(lats[i], lons[i], sides[i], type='round'))
    squares.append(create_poly(lats[i], lons[i], sides[i], type='square'))

gdf_circle = gpd.GeoDataFrame(dataset, crs='EPSG:4326', geometry=circles)
gdf_square = gpd.GeoDataFrame(dataset, crs='EPSG:4326', geometry=squares)
    

#### Four-week EY extraction

Original cell 51.
The loop selects `gdf_circle` although its destination is named `square_4w`.
The window is the 28 days before harvest, excluding the harvest date.
Its `LAI` and `NDVI` selections conflict with the intervening helper redefinitions to `LAI_mean` and `NDVI_mean`.
A selected rerun needs an explicit helper setup; this historical branch has not been normalized.

In [ ]:
# Extracts remote sensing metrics for each row in the EY dataset
for i in range(len(gdf_square))[:]:
      
    gdf = gdf_circle.iloc[i:i+1].reset_index(drop=True)
    regions = geemap.gdf_to_ee(gdf).geometry()

    jaxa = ee.Image('projects/ee-sonle96/assets/LULC_VN').clip(regions)
    mask_jaxa = jaxa.select('b1').eq(3) # b1 == 3 is the "rice" class

    name = gdf['id'][0]
    date = gdf['Date of Harvest'][0]
    print(name)

    START_DATE = (datetime.datetime.strptime(date, '%d-%m-%Y') - datetime.timedelta(28)).strftime('%Y-%m-%d')
    STOP_DATE = datetime.datetime.strptime(date, '%d-%m-%Y').strftime('%Y-%m-%d')

    # RVI
    s1 = ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT')\
              .filter(ee.Filter.eq('instrumentMode', 'IW'))\
              .filter(ee.Filter.eq('resolution_meters', 10)) \
              .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))\
              .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))\
              .filter(ee.Filter.eq('platform_number', PLATFORM_NUMBER))\
              .filter(ee.Filter.eq('orbitProperties_pass', ORBIT))\
              .filterDate(START_DATE, STOP_DATE)\
              .select(['VV', 'VH', 'angle'])


    # Speckle filter
    s1_1 = ee.ImageCollection(sf.MonoTemporal_Filter(s1,
                                                      SPECKLE_FILTER_KERNEL_SIZE,
                                                      SPECKLE_FILTER))

    # Terrain correction
    s1_1 = (trf.slope_correction(s1_1,
                                  TERRAIN_FLATTENING_MODEL,
                                  DEM,
                                  TERRAIN_FLATTENING_ADDITIONAL_LAYOVER_SHADOW_BUFFER))

    # RVI

    dprvi = s1_1.map(calculate_dprvi)\
                .mean()\
                .updateMask(mask_jaxa)

    rvi = s1_1.map(calculate_rvi)\
                .mean()\
                .updateMask(mask_jaxa)
      
    vvvh = s1_1.map(calculate_vvvh)\
                .mean()\
                .updateMask(mask_jaxa)
      

    vhvv = s1_1.map(calculate_vhvv)\
                .mean()\
                .updateMask(mask_jaxa)
      
    vv_n_vh = s1_1.mean()\
                .updateMask(mask_jaxa)


    s2 = ee.ImageCollection('COPERNICUS/S2_HARMONIZED')\
            .filterDate(START_DATE, STOP_DATE)\
            .map(mask_crop_map)

      # https://kaflekrishna.com.np/blog-detail/retrieving-leaf-area-index-lai-sentinel-2-image-google-earth-engine-gee/
    lai = s2.map(getEVI)\
            .map(getLAI)\
            .mean()\
            .select('LAI')
      
      # https://kaflekrishna.com.np/blog-detail/calculating-ndvi-sentinel-2-images/
    ndvi = s2.map(getNDVI)\
             .mean()\
             .select('NDVI')

      
    combined = vv_n_vh.addBands(dprvi)\
                      .addBands(rvi)\
                      .addBands(vvvh)\
                      .addBands(vhvv)\
                      .addBands(lai)\
                      .addBands(ndvi)

    reducers = ee.Reducer.mean().combine(
        reducer2=ee.Reducer.max(),
        sharedInputs=True
    ).combine(
        reducer2=ee.Reducer.min(),
        sharedInputs=True
    ).combine(
        reducer2=ee.Reducer.stdDev(),
        sharedInputs=True
    ).combine(
        reducer2=ee.Reducer.median(),
        sharedInputs=True
    )

    stats = combined.reduceRegion(
                      geometry=regions,
                      reducer=reducers,
                      scale=10,  
                      crs='EPSG:4326', 
                      tileScale=14,
                      bestEffort=True
                )

    try:
        test_d = stats.getInfo()
        test_d['id'] = str(name)
        test_d['date'] = date
        with open(f'datasets/EY/square_4w/{name}.json', 'w') as out_file:
          json.dump(test_d, out_file)
    except Exception as e:
         print(e)
         continue

### ADB Thai Binh study

#### Thai Binh geometries and masks

Original cell 55.
The default 2.5 m boxes extend north and east from each coordinate rather than being centred on it.
These helpers mask negative NDVI and restrict RVI/DPRVI to 0-1, unlike the rice-grid branch.
The separate QA60 function masks cloud/cirrus flags and divides optical bands by 10000.

In [ ]:
def meters_to_degrees_wgs84(latitude, meters):
    """Converts latitude from meter to latitude and longitude to degrees"""

    # 1 degree latitude ≈ 111.32 km or 111320 meters
    lat_degree = meters / 111320
    
    # 1 degree longitude ≈ 111.32 km * cos(latitude), where latitude is in radians
    lon_degree = meters / (111320 * math.cos(math.radians(latitude)))
    
    return lat_degree, lon_degree

def create_bounding_box(lat, lon, side_length_m=2.5):
    """Creates a bounding box around a point with a given side length in meters"""

    # Convert side length in meters to degrees
    lat_delta, lon_delta = meters_to_degrees_wgs84(lat, side_length_m)
    
    # Calculate min and max for latitude and longitude (bounding box)
    min_lat = lat
    max_lat = lat + lat_delta
    min_lon = lon
    max_lon = lon + lon_delta
    
    return [min_lon, min_lat, max_lon, max_lat]


def calculate_area_from_bbox(bbox):
    """Computes the area in square meters from a bounding box"""
    min_lon, min_lat, max_lon, max_lat = bbox
    
    # Calculate the latitude and longitude differences in degrees
    delta_lat = max_lat - min_lat
    delta_lon = max_lon - min_lon
    
    # Convert the latitude difference to meters (latitude is constant)
    lat_distance_meters = delta_lat * 111320  # 1 degree latitude ≈ 111.32 km
    
    # Convert the longitude difference to meters (varies with latitude)
    # Use the average latitude for more accurate calculation
    avg_lat = (min_lat + max_lat) / 2
    lon_distance_meters = delta_lon * (111320 * math.cos(math.radians(avg_lat)))
    
    # Calculate the area in square meters
    area_m2 = lat_distance_meters * lon_distance_meters
    return area_m2


def mask_s2_clouds(image):
  """Masks clouds in a Sentinel-2 image using the QA band.

  Args:
      image (ee.Image): A Sentinel-2 image.

  Returns:
      ee.Image: A cloud-masked Sentinel-2 image.
  """
  qa = image.select('QA60')

  # Bits 10 and 11 are clouds and cirrus, respectively.
  cloud_bit_mask = 1 << 10
  cirrus_bit_mask = 1 << 11

  # Both flags should be set to zero, indicating clear conditions.
  mask = (
      qa.bitwiseAnd(cloud_bit_mask)
      .eq(0)
      .And(qa.bitwiseAnd(cirrus_bit_mask).eq(0))
  )

  return image.updateMask(mask).divide(10000)


def getNDVI(image):

    # Normalized difference vegetation index (NDVI)
    ndvi = image.normalizedDifference(['B8','B4']).rename("NDVI")

    # Masks values below 0 
    ndvi = ndvi.updateMask(ndvi.gte(0))

    return(ndvi)

def calculate_rvi(image):
  """Calculates the Radar Vegetation Index (RVI) from a Sentinel-1 image using the formula in the paper below."""
  # https://www.sciencedirect.com/science/article/abs/pii/S0924271621001441?via%3Dihub

  n = image.select('VH').multiply(4)
  d = image.select('VV').add(image.select('VH'))
  rvi = n.divide(d).rename('RVI')

  # Mask values outside the valid range (0 ≤ RVI ≤ 1)
  rvi = rvi.updateMask(rvi.gte(0).And(rvi.lte(1)))

  return rvi

def calculate_dprvi(image):
  # Calculates the RVI using the following formula:
  # https://github.com/Narayana-Rao/dual_pol_descriptors/blob/main/Jupyter%20Notebooks/1.Dual-pol%20descriptors.ipynb
  q = image.select('VH').divide(image.select('VV'))
  q = q.where(q.lt(0), 0)
  q = q.where(q.gt(1), 1)
  N = q.multiply(q.add(3))
  D = (q.add(1)).multiply(q.add(1))
  dprvi = N.divide(D).rename('DPRVI')

  # Mask values outside the valid range (0 ≤ RVI ≤ 1)
  dprvi = dprvi.updateMask(dprvi.gte(0).And(dprvi.lte(1)))

  return dprvi

def calculate_vvvh(image):
  """Calculates the ratio of VV to VH bands."""
  vvvh = image.select('VV').divide(image.select('VH')).rename('VVVH')
  return vvvh

def calculate_vhvv(image):
  """Calculates the ratio of VH to VV bands."""
  vhvv = image.select('VH').divide(image.select('VV')).rename('VHVV')
  return vhvv

#### Load Thai Binh observations

Original cell 56.
This cell reads the local CSV, constructs boxes, normalizes date separators, and resets SAR parameters.
Later cells parse dates as day-month-year.
Its initial four-week lookback is overwritten by subsequent loops.

In [ ]:
# Loading the ADB Thai Binh insitu dataset 
df = pd.read_csv(r'D:\Work\ADB\SAR\datasets\thai_binh_insitu.csv')

df['bounds'] = df.apply(lambda row: create_bounding_box(row['latitude'], row['longitude']), axis=1)
df['area'] = df.apply(lambda row: calculate_area_from_bbox(row['bounds']), axis=1)
df['harvest_date'] = df['harvest_date'].str.replace('/', '-')

# Extracting bounding box and harvest date data 
bboxes = df['bounds'].to_list()
dates = df['harvest_date'].to_list()

week_lookback = 4
days_lookback = 7*week_lookback

# Parameters for SAR Processing
APPLY_BORDER_NOISE_CORRECTION = False
APPLY_TERRAIN_FLATTENING = True
APPLY_SPECKLE_FILTERING = True
POLARIZATION = 'VVVH'
PLATFORM_NUMBER = 'A'
ORBIT = 'DESCENDING'
ORBIT_NUM = None
SPECKLE_FILTER_FRAMEWORK = 'MULTI'
SPECKLE_FILTER = 'LEE SIGMA'
SPECKLE_FILTER_KERNEL_SIZE = 3 # 7
SPECKLE_FILTER_NR_OF_IMAGES = 3 # 10
TERRAIN_FLATTENING_MODEL = 'VOLUME'
DEM = ee.Image('USGS/SRTMGL1_003')
TERRAIN_FLATTENING_ADDITIONAL_LAYOVER_SHADOW_BUFFER = 0
CLIP_TO_ROI = True

#### Write field-box geometries

Original cell 57.
This cell writes a GeoJSON in the original `shapefiles` directory.
It does not extract satellite values.

In [ ]:
from shapely.geometry import box

gdf = gpd.GeoDataFrame({'plot': df.index}, crs='EPSG:4326', geometry=[box(*bbox) for bbox in bboxes])
gdf.to_file(r"D:\Work\ADB\SAR\shapefiles/thai_binh_insitu.json", driver='GeoJSON', index=False)

#### Write the administrative subset

Original cell 58.
The admin-2 filter uses the source spelling `TháiBình`.
Review that value if using another boundary release.

In [ ]:
poly = gpd.read_file(r"D:\Work\ADB\SAR\shapefiles\gadm41_VNM_2.json")
poly[poly['NAME_1'] == 'TháiBình'].to_file(r"D:\Work\ADB\SAR\shapefiles/thai_binh_province.json", driver='GeoJSON', index=False)

#### Inspect harvest-date strings

Original cell 59.
This display checks the strings consumed by the day-month-year parser.
The previous display output was cleared.

In [ ]:
df['harvest_date'].unique()

#### One-to-eight-week field summaries

Original cell 61.
Each cumulative lookback combines temporal index summaries with spatial mean/max/min reducers at nominal 10 m.
This variant applies QA60 cloud masking, writes JSONs under `datasets/thai_binh/{week_lookback}w`, and continues after exceptions.

In [ ]:
for week_lookback in range(1, 9, 1):
    print(week_lookback)
    days_lookback = 7*week_lookback
    # Looping over each entry in the ADB dataset 
    for i in tqdm_notebook(range(len(bboxes))):
        bbox = bboxes[i]
        ROI = ee.Geometry.BBox(bbox[0],
                                bbox[1],
                                bbox[2],
                                bbox[3])\
                        .bounds(proj='EPSG:4326')
        
        date = dates[i]

        START_DATE = (datetime.datetime.strptime(date, '%d-%m-%Y') - datetime.timedelta(days_lookback)).strftime('%Y-%m-%d')
        STOP_DATE = datetime.datetime.strptime(date, '%d-%m-%Y').strftime('%Y-%m-%d')

        # RVI
        s1 = ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT')\
                .filter(ee.Filter.eq('instrumentMode', 'IW'))\
                .filter(ee.Filter.eq('resolution_meters', 10)) \
                .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))\
                .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))\
                .filter(ee.Filter.eq('platform_number', PLATFORM_NUMBER))\
                .filter(ee.Filter.eq('orbitProperties_pass', ORBIT))\
                .filterDate(START_DATE, STOP_DATE)\
                .select(['VV', 'VH', 'angle'])


        # Speckle filter
        s1_1 = ee.ImageCollection(sf.MonoTemporal_Filter(s1,
                                                        SPECKLE_FILTER_KERNEL_SIZE,
                                                        SPECKLE_FILTER))

        # Terrain correction
        s1_1 = (trf.slope_correction(s1_1,
                                    TERRAIN_FLATTENING_MODEL,
                                    DEM,
                                    TERRAIN_FLATTENING_ADDITIONAL_LAYOVER_SHADOW_BUFFER))

        # RVI

        dprvi_mean = s1_1.map(calculate_dprvi)\
                    .mean()\
                    .rename('DPRVI_mean')
        
        dprvi_max = s1_1.map(calculate_dprvi)\
                    .max()\
                    .rename('DPRVI_max')
        
        dprvi_min = s1_1.map(calculate_dprvi)\
                    .min()\
                    .rename('DPRVI_min')

        rvi_mean = s1_1.map(calculate_rvi)\
                    .mean()\
                    .rename("RVI_mean")
                    
        rvi_max = s1_1.map(calculate_rvi)\
                    .max()\
                    .rename("RVI_max")
                    
        rvi_min = s1_1.map(calculate_rvi)\
                    .min()\
                    .rename("RVI_min")
        
        vvvh_mean = s1_1.map(calculate_vvvh)\
                    .mean()\
                    .rename('VVVH_mean')
        
        vvvh_max = s1_1.map(calculate_vvvh)\
                    .max()\
                    .rename('VVVH_max')
        
        vvvh_min = s1_1.map(calculate_vvvh)\
                    .min()\
                    .rename('VVVH_min')
        

        vhvv_mean = s1_1.map(calculate_vhvv)\
                    .mean()\
                    .rename("VHVV_mean")
        
        vhvv_max = s1_1.map(calculate_vhvv)\
                    .max()\
                    .rename("VHVV_max")
        
        vhvv_min = s1_1.map(calculate_vhvv)\
                    .min()\
                    .rename("VHVV_min")
        


        s2 = ee.ImageCollection('COPERNICUS/S2_HARMONIZED')\
                .filterDate(START_DATE, STOP_DATE)\
                .map(mask_s2_clouds)
                # .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))\
                
        
        # https://kaflekrishna.com.np/blog-detail/calculating-ndvi-sentinel-2-images/
        ndvi_mean = s2.map(getNDVI)\
                    .select('NDVI')\
                    .mean()\
                    .rename('NDVI_mean')
        
        # https://kaflekrishna.com.np/blog-detail/calculating-ndvi-sentinel-2-images/
        ndvi_max = s2.map(getNDVI)\
                    .select('NDVI')\
                    .max()\
                    .rename('NDVI_max')
        
        # https://kaflekrishna.com.np/blog-detail/calculating-ndvi-sentinel-2-images/
        ndvi_min = s2.map(getNDVI)\
                    .select('NDVI')\
                    .min()\
                    .rename('NDVI_min')

        
        combined = dprvi_mean.addBands(dprvi_max)\
                            .addBands(dprvi_min)\
                            .addBands(rvi_mean)\
                            .addBands(rvi_max)\
                            .addBands(rvi_min)\
                            .addBands(vvvh_mean)\
                            .addBands(vvvh_max)\
                            .addBands(vvvh_min)\
                            .addBands(vhvv_mean)\
                            .addBands(vhvv_max)\
                            .addBands(vhvv_min)\
                            .addBands(ndvi_mean)\
                            .addBands(ndvi_max)\
                            .addBands(ndvi_min)

        # combined = ndvi_mean.addBands(ndvi_max).addBands(ndvi_min)

        reducers = ee.Reducer.mean().combine(
            reducer2=ee.Reducer.max(),
            sharedInputs=True
        ).combine(
            reducer2=ee.Reducer.min(),
            sharedInputs=True
        )

        stats = combined.reduceRegion(
                        geometry=ROI,
                        reducer=reducers,
                        scale=10,  
                        crs='EPSG:4326', 
                        tileScale=14,
                        bestEffort=True
                    )

        try:
            test_d = stats.getInfo()
            test_d['date'] = date
            with open(f'datasets/thai_binh/{week_lookback}w/{i}_ndvi_rvi_ver2.json', 'w') as out_file:
                json.dump(test_d, out_file)
        except Exception as e:
            print(e)
            continue
        

#### Six-to-four-week pre-harvest interval

Original cell 62.
This experiment uses the interval from 42 to 28 days before harvest rather than a cumulative window.
Its `4w6w` destination separates it from other summaries; it uses the same cloud and range masks.

In [ ]:

start_lookback = 42
end_lookback = 28
# Looping over each entry in the ADB dataset 
for i in tqdm_notebook(range(len(bboxes))):
    bbox = bboxes[i]
    ROI = ee.Geometry.BBox(bbox[0],
                                bbox[1],
                                bbox[2],
                                bbox[3])\
                        .bounds(proj='EPSG:4326')
        
    date = dates[i]

    START_DATE = (datetime.datetime.strptime(date, '%d-%m-%Y') - datetime.timedelta(start_lookback)).strftime('%Y-%m-%d')
    STOP_DATE = (datetime.datetime.strptime(date, '%d-%m-%Y') - datetime.timedelta(end_lookback)).strftime('%Y-%m-%d')

    # RVI
    s1 = ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT')\
                .filter(ee.Filter.eq('instrumentMode', 'IW'))\
                .filter(ee.Filter.eq('resolution_meters', 10)) \
                .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))\
                .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))\
                .filter(ee.Filter.eq('platform_number', PLATFORM_NUMBER))\
                .filter(ee.Filter.eq('orbitProperties_pass', ORBIT))\
                .filterDate(START_DATE, STOP_DATE)\
                .select(['VV', 'VH', 'angle'])


    # Speckle filter
    s1_1 = ee.ImageCollection(sf.MonoTemporal_Filter(s1,
                                                        SPECKLE_FILTER_KERNEL_SIZE,
                                                        SPECKLE_FILTER))

    # Terrain correction
    s1_1 = (trf.slope_correction(s1_1,
                                    TERRAIN_FLATTENING_MODEL,
                                    DEM,
                                    TERRAIN_FLATTENING_ADDITIONAL_LAYOVER_SHADOW_BUFFER))

    # RVI

    dprvi_mean = s1_1.map(calculate_dprvi)\
                    .mean()\
                    .rename('DPRVI_mean')

    rvi_mean = s1_1.map(calculate_rvi)\
                    .mean()\
                    .rename("RVI_mean")
                
        
    vvvh_mean = s1_1.map(calculate_vvvh)\
                    .mean()\
                    .rename('VVVH_mean')
        

    vhvv_mean = s1_1.map(calculate_vhvv)\
                    .mean()\
                    .rename("VHVV_mean")
        

    s2 = ee.ImageCollection('COPERNICUS/S2_HARMONIZED')\
                .filterDate(START_DATE, STOP_DATE)\
                .map(mask_s2_clouds)
                # .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))\
                
        
    # https://kaflekrishna.com.np/blog-detail/calculating-ndvi-sentinel-2-images/
    ndvi_mean = s2.map(getNDVI)\
                    .select('NDVI')\
                    .mean()\
                    .rename('NDVI_mean')

        
    combined = dprvi_mean.addBands(rvi_mean)\
                        .addBands(vvvh_mean)\
                        .addBands(vhvv_mean)\
                        .addBands(ndvi_mean)\

    # combined = ndvi_mean.addBands(ndvi_max).addBands(ndvi_min)

    reducers = ee.Reducer.mean().combine(
            reducer2=ee.Reducer.max(),
            sharedInputs=True
        ).combine(
            reducer2=ee.Reducer.min(),
            sharedInputs=True
        )

    stats = combined.reduceRegion(
                        geometry=ROI,
                        reducer=reducers,
                        scale=10,  
                        crs='EPSG:4326', 
                        tileScale=14,
                        bestEffort=True
                    )

    try:
        test_d = stats.getInfo()
        test_d['date'] = date
        with open(f'datasets/thai_binh/4w6w/{i}_ndvi_rvi_ver2.json', 'w') as out_file:
            json.dump(test_d, out_file)
    except Exception as e:
        print(e)
        continue
        

#### Four-week unbuffered raster exports

Original cell 65.
The active variant uses the original field box and applies the optical cloud mask.
It submits five float32 index bands to `raster_no_buffer`.
Later alternatives use different cloud handling and can reuse the same export names.

In [ ]:
for week_lookback in range(4, 5, 1):
    print(week_lookback)
    days_lookback = 7*week_lookback
    # Looping over each entry in the ADB dataset 
    for i in tqdm_notebook(range(len(bboxes))[:]):
        bbox = bboxes[i]

        ROI = ee.Geometry.BBox(bbox[0],
                                bbox[1],
                                bbox[2],
                                bbox[3])\
                        .bounds(proj='EPSG:4326')
        
        # # Create the bounding geometry
        # roi = ee.Geometry.BBox(*bbox)

        # # Convert to a projected CRS (e.g., UTM zone for Vietnam - EPSG:32648 or 32649 depending on location)
        # # We'll use EPSG:32648 (UTM zone 48N) as this bbox is near 106°E
        # projected = roi.transform('EPSG:32648', 0.001)

        # # Buffer the projected geometry by 5 meters
        # buffered_projected = projected.buffer(5)

        # # Reproject the buffered geometry back to EPSG:4326
        # buffered_roi = buffered_projected.transform('EPSG:4326', 0.00001).getInfo()

        # ROI = ee.Geometry.BBox(buffered_roi[0],
        #                         buffered_roi[1],
        #                         buffered_roi[2],
        #                         buffered_roi[3])\
        #                 .bounds(proj='EPSG:4326')
        
        date = dates[i]

        START_DATE = (datetime.datetime.strptime(date, '%d-%m-%Y') - datetime.timedelta(days_lookback)).strftime('%Y-%m-%d')
        STOP_DATE = datetime.datetime.strptime(date, '%d-%m-%Y').strftime('%Y-%m-%d')

        # RVI
        s1 = ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT')\
                .filter(ee.Filter.eq('instrumentMode', 'IW'))\
                .filter(ee.Filter.eq('resolution_meters', 10)) \
                .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))\
                .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))\
                .filter(ee.Filter.eq('platform_number', PLATFORM_NUMBER))\
                .filter(ee.Filter.eq('orbitProperties_pass', ORBIT))\
                .filterDate(START_DATE, STOP_DATE)\
                .select(['VV', 'VH', 'angle'])


        # Speckle filter
        s1_1 = ee.ImageCollection(sf.MonoTemporal_Filter(s1,
                                                        SPECKLE_FILTER_KERNEL_SIZE,
                                                        SPECKLE_FILTER))

        # Terrain correction
        s1_1 = (trf.slope_correction(s1_1,
                                    TERRAIN_FLATTENING_MODEL,
                                    DEM,
                                    TERRAIN_FLATTENING_ADDITIONAL_LAYOVER_SHADOW_BUFFER))

        # RVI

        dprvi_mean = s1_1.map(calculate_dprvi)\
                    .mean()\
                    .rename('DPRVI_mean')\
                    .toFloat()
        

        rvi_mean = s1_1.map(calculate_rvi)\
                    .mean()\
                    .rename("RVI_mean")\
                    .toFloat()
        
        vvvh_mean = s1_1.map(calculate_vvvh)\
                    .mean()\
                    .rename('VVVH_mean')\
                    .toFloat()
        

        vhvv_mean = s1_1.map(calculate_vhvv)\
                    .mean()\
                    .rename("VHVV_mean")\
                    .toFloat()
        

        s2 = ee.ImageCollection('COPERNICUS/S2_HARMONIZED')\
                .filterDate(START_DATE, STOP_DATE)\
                .map(mask_s2_clouds)
                # .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))\
                
        
        # https://kaflekrishna.com.np/blog-detail/calculating-ndvi-sentinel-2-images/
        ndvi_mean = s2.map(getNDVI)\
                    .select('NDVI')\
                    .mean()\
                    .rename('NDVI_mean')\
                    .toFloat()

        
        combined = dprvi_mean.addBands(rvi_mean)\
                            .addBands(vvvh_mean)\
                            .addBands(vhvv_mean)\
                            .addBands(ndvi_mean)\

        # combined = ndvi_mean.addBands(ndvi_max).addBands(ndvi_min)

        # reducers = ee.Reducer.mean().combine(
        #     reducer2=ee.Reducer.max(),
        #     sharedInputs=True
        # ).combine(
        #     reducer2=ee.Reducer.min(),
        #     sharedInputs=True
        # )

        # stats = combined.reduceRegion(
        #                 geometry=ROI,
        #                 reducer=reducers,
        #                 scale=10,  
        #                 crs='EPSG:4326', 
        #                 tileScale=14,
        #                 bestEffort=True
        #             )

        # try:
        #     test_d = stats.getInfo()
        #     test_d['date'] = date
        #     with open(f'datasets/thai_binh/{week_lookback}w/{i}_ndvi_rvi_ver2.json', 'w') as out_file:
        #         json.dump(test_d, out_file)
        # except Exception as e:
        #     print(e)
        #     continue

        # Export image to Google Drive
        export_task = ee.batch.Export.image.toDrive(
            image=combined,
            description=f'{i}_week{week_lookback}',
            folder='raster_no_buffer',
            fileNamePrefix=f'{i}_week{week_lookback}',
            region=ROI,
            scale=10,
            crs='EPSG:4326',
            maxPixels=1e13
        )
        export_task.start()

        
        

#### Buffered four-to-eight-week rasters

Original cell 66.
The code projects the field box to EPSG:32648, adds a 5 m buffer, and takes its bounding rectangle in EPSG:4326.
The optical cloud-mask call is commented out.
Exports go to `raster_buffer`.

In [ ]:
for week_lookback in range(4, 9, 1):
    print(week_lookback)
    days_lookback = 7*week_lookback
    # Looping over each entry in the ADB dataset 
    for i in tqdm_notebook(range(len(bboxes))[:]):
        bbox = bboxes[i]

        # ROI = ee.Geometry.BBox(bbox[0],
        #                         bbox[1],
        #                         bbox[2],
        #                         bbox[3])\
        #                 .bounds(proj='EPSG:4326')
        
        # Create the bounding geometry
        roi = ee.Geometry.BBox(*bbox)

        # Convert to a projected CRS (e.g., UTM zone for Vietnam - EPSG:32648 or 32649 depending on location)
        # We'll use EPSG:32648 (UTM zone 48N) as this bbox is near 106°E
        projected = roi.transform('EPSG:32648', 0.001)

        # Buffer the projected geometry by 5 meters
        buffered_projected = projected.buffer(5)

        # Reproject the buffered geometry back to EPSG:4326
        buffered_roi = buffered_projected.transform('EPSG:4326', 0.001)

        # ROI = ee.Geometry.BBox(buffered_roi[0],
        #                         buffered_roi[1],
        #                         buffered_roi[2],
        #                         buffered_roi[3])\
        #                 .bounds(proj='EPSG:4326')
        ROI = buffered_roi.bounds()
        
        date = dates[i]

        START_DATE = (datetime.datetime.strptime(date, '%d-%m-%Y') - datetime.timedelta(days_lookback)).strftime('%Y-%m-%d')
        STOP_DATE = datetime.datetime.strptime(date, '%d-%m-%Y').strftime('%Y-%m-%d')

        # RVI
        s1 = ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT')\
                .filter(ee.Filter.eq('instrumentMode', 'IW'))\
                .filter(ee.Filter.eq('resolution_meters', 10)) \
                .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))\
                .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))\
                .filter(ee.Filter.eq('platform_number', PLATFORM_NUMBER))\
                .filter(ee.Filter.eq('orbitProperties_pass', ORBIT))\
                .filterDate(START_DATE, STOP_DATE)\
                .select(['VV', 'VH', 'angle'])


        # Speckle filter
        s1_1 = ee.ImageCollection(sf.MonoTemporal_Filter(s1,
                                                        SPECKLE_FILTER_KERNEL_SIZE,
                                                        SPECKLE_FILTER))

        # Terrain correction
        s1_1 = (trf.slope_correction(s1_1,
                                    TERRAIN_FLATTENING_MODEL,
                                    DEM,
                                    TERRAIN_FLATTENING_ADDITIONAL_LAYOVER_SHADOW_BUFFER))

        # RVI

        dprvi_mean = s1_1.map(calculate_dprvi)\
                    .mean()\
                    .rename('DPRVI_mean')\
                    .toFloat()
        

        rvi_mean = s1_1.map(calculate_rvi)\
                    .mean()\
                    .rename("RVI_mean")\
                    .toFloat()
        
        vvvh_mean = s1_1.map(calculate_vvvh)\
                    .mean()\
                    .rename('VVVH_mean')\
                    .toFloat()
        

        vhvv_mean = s1_1.map(calculate_vhvv)\
                    .mean()\
                    .rename("VHVV_mean")\
                    .toFloat()
        

        s2 = ee.ImageCollection('COPERNICUS/S2_HARMONIZED')\
                .filterDate(START_DATE, STOP_DATE)\
                # .map(mask_s2_clouds)
                # .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))\
                
        
        # https://kaflekrishna.com.np/blog-detail/calculating-ndvi-sentinel-2-images/
        ndvi_mean = s2.map(getNDVI)\
                    .select('NDVI')\
                    .mean()\
                    .rename('NDVI_mean')\
                    .toFloat()

        
        combined = dprvi_mean.addBands(rvi_mean)\
                            .addBands(vvvh_mean)\
                            .addBands(vhvv_mean)\
                            .addBands(ndvi_mean)\

        # combined = ndvi_mean.addBands(ndvi_max).addBands(ndvi_min)

        # reducers = ee.Reducer.mean().combine(
        #     reducer2=ee.Reducer.max(),
        #     sharedInputs=True
        # ).combine(
        #     reducer2=ee.Reducer.min(),
        #     sharedInputs=True
        # )

        # stats = combined.reduceRegion(
        #                 geometry=ROI,
        #                 reducer=reducers,
        #                 scale=10,  
        #                 crs='EPSG:4326', 
        #                 tileScale=14,
        #                 bestEffort=True
        #             )

        # try:
        #     test_d = stats.getInfo()
        #     test_d['date'] = date
        #     with open(f'datasets/thai_binh/{week_lookback}w/{i}_ndvi_rvi_ver2.json', 'w') as out_file:
        #         json.dump(test_d, out_file)
        # except Exception as e:
        #     print(e)
        #     continue

        # Export image to Google Drive
        export_task = ee.batch.Export.image.toDrive(
            image=combined,
            description=f'{i}_week{week_lookback}',
            folder='raster_buffer',
            fileNamePrefix=f'{i}_week{week_lookback}',
            region=ROI,
            scale=10,
            crs='EPSG:4326',
            maxPixels=1e13
        )
        export_task.start()

        



#### Unbuffered four-to-eight-week rasters

Original cell 67.
This alternative uses the original box and leaves optical cloud masking commented out.
Its `raster_no_buffer` exports reuse four-week names from the earlier cloud-masked variant.
Distinguish these variants when interpreting old exports.

In [ ]:
for week_lookback in range(4, 9, 1):
    print(week_lookback)
    days_lookback = 7*week_lookback
    # Looping over each entry in the ADB dataset 
    for i in tqdm_notebook(range(len(bboxes))[:]):
        bbox = bboxes[i]

        ROI = ee.Geometry.BBox(bbox[0],
                                bbox[1],
                                bbox[2],
                                bbox[3])\
                        .bounds(proj='EPSG:4326')
        
        # # Create the bounding geometry
        # roi = ee.Geometry.BBox(*bbox)

        # # Convert to a projected CRS (e.g., UTM zone for Vietnam - EPSG:32648 or 32649 depending on location)
        # # We'll use EPSG:32648 (UTM zone 48N) as this bbox is near 106°E
        # projected = roi.transform('EPSG:32648', 0.001)

        # # Buffer the projected geometry by 5 meters
        # buffered_projected = projected.buffer(5)

        # # Reproject the buffered geometry back to EPSG:4326
        # buffered_roi = buffered_projected.transform('EPSG:4326', 0.001)

        # ROI = buffered_roi.bounds()
        
        date = dates[i]

        START_DATE = (datetime.datetime.strptime(date, '%d-%m-%Y') - datetime.timedelta(days_lookback)).strftime('%Y-%m-%d')
        STOP_DATE = datetime.datetime.strptime(date, '%d-%m-%Y').strftime('%Y-%m-%d')

        # RVI
        s1 = ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT')\
                .filter(ee.Filter.eq('instrumentMode', 'IW'))\
                .filter(ee.Filter.eq('resolution_meters', 10)) \
                .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))\
                .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))\
                .filter(ee.Filter.eq('platform_number', PLATFORM_NUMBER))\
                .filter(ee.Filter.eq('orbitProperties_pass', ORBIT))\
                .filterDate(START_DATE, STOP_DATE)\
                .select(['VV', 'VH', 'angle'])


        # Speckle filter
        s1_1 = ee.ImageCollection(sf.MonoTemporal_Filter(s1,
                                                        SPECKLE_FILTER_KERNEL_SIZE,
                                                        SPECKLE_FILTER))

        # Terrain correction
        s1_1 = (trf.slope_correction(s1_1,
                                    TERRAIN_FLATTENING_MODEL,
                                    DEM,
                                    TERRAIN_FLATTENING_ADDITIONAL_LAYOVER_SHADOW_BUFFER))

        # RVI

        dprvi_mean = s1_1.map(calculate_dprvi)\
                    .mean()\
                    .rename('DPRVI_mean')\
                    .toFloat()
        

        rvi_mean = s1_1.map(calculate_rvi)\
                    .mean()\
                    .rename("RVI_mean")\
                    .toFloat()
        
        vvvh_mean = s1_1.map(calculate_vvvh)\
                    .mean()\
                    .rename('VVVH_mean')\
                    .toFloat()
        

        vhvv_mean = s1_1.map(calculate_vhvv)\
                    .mean()\
                    .rename("VHVV_mean")\
                    .toFloat()
        

        s2 = ee.ImageCollection('COPERNICUS/S2_HARMONIZED')\
                .filterDate(START_DATE, STOP_DATE)\
                # .map(mask_s2_clouds)
                # .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))\
                
        
        # https://kaflekrishna.com.np/blog-detail/calculating-ndvi-sentinel-2-images/
        ndvi_mean = s2.map(getNDVI)\
                    .select('NDVI')\
                    .mean()\
                    .rename('NDVI_mean')\
                    .toFloat()

        
        combined = dprvi_mean.addBands(rvi_mean)\
                            .addBands(vvvh_mean)\
                            .addBands(vhvv_mean)\
                            .addBands(ndvi_mean)\

        # Export image to Google Drive
        export_task = ee.batch.Export.image.toDrive(
            image=combined,
            description=f'{i}_week{week_lookback}',
            folder='raster_no_buffer',
            fileNamePrefix=f'{i}_week{week_lookback}',
            region=ROI,
            scale=10,
            crs='EPSG:4326',
            maxPixels=1e13
        )
        export_task.start()

        




#### Weather for the current field lookback

Original cell 71.
This cell inherits `week_lookback` and `days_lookback`; after the preceding loop they are eight weeks and 56 days.
CAMS is commented out, and `mean_temp` is added twice, producing a suffixed duplicate band.
Weather JSONs are written beside the other lookback outputs.

In [ ]:
# Looping over each entry in the ADB dataset 
for i in tqdm_notebook(range(len(bboxes))):
    bbox = bboxes[i]
    ROI = ee.Geometry.BBox(bbox[0],
                            bbox[1],
                            bbox[2],
                            bbox[3])\
                    .bounds(proj='EPSG:4326')
    
    date = dates[i]

    START_DATE = (datetime.datetime.strptime(date, '%d-%m-%Y') - datetime.timedelta(days_lookback)).strftime('%Y-%m-%d')
    STOP_DATE = datetime.datetime.strptime(date, '%d-%m-%Y').strftime('%Y-%m-%d')

    # pm25 = ee.ImageCollection("ECMWF/CAMS/NRT")\
    #           .filter(ee.Filter.date(START_DATE, STOP_DATE))

    # pm25_mean = pm25.select('particulate_matter_d_less_than_25_um_surface')\
    #                     .mean()\
    #                     .multiply(1E9)\
    #                     .rename('PM25_mean')
        
    # pm25_min = pm25.select('particulate_matter_d_less_than_25_um_surface')\
    #                 .min()\
    #                 .multiply(1E9)\
    #                 .rename('PM25_min')
        
    # pm25_max = pm25.select('particulate_matter_d_less_than_25_um_surface')\
    #                 .max()\
    #                 .multiply(1E9)\
    #                 .rename('PM25_max')
        
    # ERA5
    era5 = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")\
                .filter(ee.Filter.date(START_DATE, STOP_DATE))

    # Average temperature
    mean_temp = era5.select('temperature_2m').mean()\
                    .subtract(273.15)\
                    .rename('Temperature_mean')

    # Max temperature
    max_temp = era5.select('temperature_2m').max()\
                    .subtract(273.15)\
                    .rename('Temperature_max')

    #   precipitation
    prep = era5.select('total_precipitation_sum').sum()\
                .rename('Precipitation_sum')

    # Relative Humidity
    # https://earthscience.stackexchange.com/questions/16570/how-to-calculate-relative-humidity-from-temperature-dew-point-and-pressure?newreg=86cf2a3bb5bb47c5afd5fcaccfea796a
    b = 17.625
    c = 243.04
    mean_dew_temp = era5.select('dewpoint_temperature_2m').mean()\
                            .subtract(273.15)\
                            .rename('avg_dew_temperature')

    num = mean_dew_temp.subtract(mean_temp).multiply(c).multiply(b)
    deno = (mean_temp.add(c)).multiply(mean_dew_temp.add(c))
    mean_rh = num.divide(deno).exp()\
                    .rename('Rel_humidity_mean')

        
    combined = mean_temp.addBands(mean_temp)\
                        .addBands(max_temp)\
                        .addBands(prep)\
                        .addBands(mean_rh)


    reducers = ee.Reducer.mean().combine(
        reducer2=ee.Reducer.max(),
        sharedInputs=True
    ).combine(
        reducer2=ee.Reducer.min(),
        sharedInputs=True
    )

    stats = combined.reduceRegion(
                      geometry=ROI,
                      reducer=reducers,
                      scale=10,  
                      crs='EPSG:4326', 
                      tileScale=14,
                      bestEffort=True
                )

    try:
        test_d = stats.getInfo()
        test_d['date'] = date
        with open(f'datasets/thai_binh/{week_lookback}w/{i}_rest.json', 'w') as out_file:
          json.dump(test_d, out_file)
    except Exception as e:
         print(e)
         continue
    

#### Legacy field ESI, FPAR, and soil moisture

Original cell 72.
This cell inherits the lookback and combines ESI, Terra/Aqua FPAR, and SMAP root-zone moisture.
The maintained [extract_channels.py](replication/extract_channels.py) follows the requested ESI/Aqua-only scope.
This older branch is retained to interpret historical field-channel JSONs.

In [ ]:
for i in tqdm_notebook(range(len(bboxes))):
    bbox = bboxes[i]
    ROI = ee.Geometry.BBox(bbox[0],
                            bbox[1],
                            bbox[2],
                            bbox[3])\
                    .bounds(proj='EPSG:4326')
    
    date = dates[i]

    START_DATE = (datetime.datetime.strptime(date, '%d-%m-%Y') - datetime.timedelta(days_lookback)).strftime('%Y-%m-%d')
    STOP_DATE = datetime.datetime.strptime(date, '%d-%m-%Y').strftime('%Y-%m-%d')

    esi_4wk_ic = ee.ImageCollection('projects/climate-engine/esi/4wk')\
                       .filterDate(START_DATE, STOP_DATE)
        
    esi_4wk_mean = esi_4wk_ic.select('ESI')\
                                 .mean()\
                                 .rename('ESI_4wk_mean')
        
    esi_4wk_min = esi_4wk_ic.select('ESI')\
                                 .min()\
                                 .rename('ESI_4wk_min')

    esi_4wk_max = esi_4wk_ic.select('ESI')\
                                 .max()\
                                 .rename('ESI_4wk_max')
    
    smap_l4_ic = ee.ImageCollection('NASA/SMAP/SPL4SMGP/007')\
                       .filterDate(START_DATE, STOP_DATE)
        
    sm_root_mean = smap_l4_ic.select('sm_rootzone')\
                                 .mean()\
                                 .rename('SM_ROOT_mean')
        
    sm_root_min = smap_l4_ic.select('sm_rootzone')\
                                 .min()\
                                 .rename('SM_ROOT_min')
        
    sm_root_max = smap_l4_ic.select('sm_rootzone')\
                                 .max()\
                                 .rename('SM_ROOT_max')
        
        
    MCD15A3H = ee.ImageCollection("MODIS/061/MCD15A3H")\
                     .filterDate(START_DATE, STOP_DATE)

    def extract_bits(image, bit_start, bit_end):
            num_bits = bit_end - bit_start + 1
            mask = (1 << num_bits) - 1
            return image.rightShift(bit_start).bitwiseAnd(mask)

    def apply_bitmask_terra(image):

            qc_band = image.select('FparLai_QC')

            # Extract values for the specific bit ranges
            # modland_qc = extract_bits(qc_band, 0, 0).eq(0)  # Bit 0 should be 0 (Good quality - main algorithm with or without saturation)
            # detector = extract_bits(qc_band, 2, 2).eq(0)    # Bit 2 should be 0
            # cloud_state = extract_bits(qc_band, 3, 4).eq(0) # Bits 3-4 should be 0 (clear)
            scf_qc = extract_bits(qc_band, 5, 7).neq(4)     # Bits 5-7 should not be 4 (Pixel not produced at all, value couldn''t be retrieved)

            # combined_mask = modland_qc.And(detector).And(cloud_state).And(scf_qc)

            # Sensor
            terra_mask = scf_qc.And(extract_bits(qc_band, 1, 1).eq(0)) 

            return image.updateMask(terra_mask)

    def apply_bitmask_aqua(image):

            qc_band = image.select('FparLai_QC')

            # Extract values for the specific bit ranges
            # modland_qc = extract_bits(qc_band, 0, 0).eq(0)  # Bit 0 should be 0 (Good quality - main algorithm with or without saturation)
            # detector = extract_bits(qc_band, 2, 2).eq(0)    # Bit 2 should be 0
            # cloud_state = extract_bits(qc_band, 3, 4).eq(0) # Bits 3-4 should be 0 (clear)
            scf_qc = extract_bits(qc_band, 5, 7).neq(4)     # Bits 5-7 should not be 4 (Pixel not produced at all, value couldn''t be retrieved)

            # combined_mask = modland_qc.And(detector).And(cloud_state).And(scf_qc)
            # aqua_mask =  combined_mask.And(extract_bits(qc_band, 1, 1).eq(1)) 
            aqua_mask =  scf_qc.And(extract_bits(qc_band, 1, 1).eq(1)) 

            return image.updateMask(aqua_mask)

    dataset_terra = MCD15A3H.map(apply_bitmask_terra)

    fpar_terra_mean = dataset_terra.select('Fpar')\
                                       .mean()\
                                       .multiply(0.01)\
                                       .rename('FPAR_Terra_mean')
        
    fpar_terra_min = dataset_terra.select('Fpar')\
                                       .min()\
                                       .multiply(0.01)\
                                       .rename('FPAR_Terra_min')
        
    fpar_terra_max = dataset_terra.select('Fpar')\
                                       .max()\
                                       .multiply(0.01)\
                                       .rename('FPAR_Terra_max')

    dataset_aqua = MCD15A3H.map(apply_bitmask_aqua)

    fpar_aqua_mean = dataset_aqua.select('Fpar')\
                                       .mean()\
                                       .multiply(0.01)\
                                       .rename('FPAR_Aqua_mean')
        
    fpar_aqua_min = dataset_aqua.select('Fpar')\
                                       .min()\
                                       .multiply(0.01)\
                                       .rename('FPAR_Aqua_min')
        
    fpar_aqua_max = dataset_aqua.select('Fpar')\
                                       .max()\
                                       .multiply(0.01)\
                                       .rename('FPAR_Aqua_max')
    

    combined = esi_4wk_mean.addBands(esi_4wk_min)\
                               .addBands(esi_4wk_max)\
                               .addBands(fpar_terra_mean)\
                               .addBands(fpar_terra_min)\
                               .addBands(fpar_terra_max)\
                               .addBands(fpar_aqua_mean)\
                               .addBands(fpar_aqua_min)\
                               .addBands(fpar_aqua_max)\
                               .addBands(sm_root_mean)\
                               .addBands(sm_root_min)\
                               .addBands(sm_root_max)


    reducers = ee.Reducer.mean().combine(
        reducer2=ee.Reducer.max(),
        sharedInputs=True
    ).combine(
        reducer2=ee.Reducer.min(),
        sharedInputs=True
    )

    stats = combined.reduceRegion(
                      geometry=ROI,
                      reducer=reducers,
                      scale=10,  
                      crs='EPSG:4326', 
                      tileScale=14,
                      bestEffort=True
                )

    try:
        test_d = stats.getInfo()
        test_d['date'] = date
        with open(f'datasets/thai_binh/{week_lookback}w/{i}_channels.json', 'w') as out_file:
          json.dump(test_d, out_file)
    except Exception as e:
         print(e)
         continue